# Lab 0: NumPy for Machine Learning

**SC 677 805 Deep Learning for Data Analytics**
M.Sc. in Statistics and Data Science, Faculty of Science, Khon Kaen University

This lab teaches **only the NumPy that the rest of the course actually uses**. Not all of NumPy.

### How every drill works / รูปแบบของทุกข้อ

1. **ตัวอย่างที่คำนวณให้ดูแล้ว** with real numbers and the answer written out. Read it first.
2. **โจทย์** one line for you to write.
3. **test** run it immediately, see PASS or what went wrong.

Nothing here is longer than one or two lines. If you find yourself writing five lines, stop and
re-read the worked example above the exercise.

**ถ้าตอบไม่ได้ ให้กลับไปดูตัวอย่างข้างบนข้อนั้น ไม่ต้องเปิดอินเทอร์เน็ต ทุกคำตอบอยู่ในตัวอย่างแล้ว**

### Scoring

This lab is graded on **completion**, not on getting everything right the first time. Run the tests as many times as you like. The score is there so you know where you stand, not to rank you.

### Sections

| | หัวข้อ | 
| --- | --- | 
| 1 | สร้าง array และมองรูปร่าง |
| 2 | คำนวณทีละตัว และ broadcasting | 
| 3 | รวมค่าด้วย axis | 
| 4 | boolean mask | 
| 5 | unique, argmax, argsort |
| 6 | พีชคณิตเชิงเส้น | 
| 7 | การสุ่ม | 
| 8 | ประกอบร่าง (ไม่คิดคะแนน) | 

## Setup

Run this cell first.

In [1]:
import numpy as np
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
print("NumPy", np.__version__)

SCORES, MAXPTS = {}, {}


def grade(name, points, *tests):
    MAXPTS[name] = points
    problems = []
    for t in tests:
        try:
            t()
        except AssertionError as e:
            problems.append("FAIL  %s: %s" % (t.__name__, str(e) or "assertion failed"))
        except Exception as e:
            problems.append("ERROR %s: %s: %s" % (t.__name__, type(e).__name__, e))
    SCORES[name] = 0 if problems else points
    print("[%s] %-32s %3d / %d" % ("PASS" if not problems else "----",
                                   name, SCORES[name], points))
    for p in problems:
        print("         " + p)


def report():
    names = {"1": "สร้าง array และรูปร่าง", "2": "elementwise และ broadcasting",
             "3": "รวมค่าด้วย axis", "4": "boolean mask",
             "5": "unique, argmax, argsort", "6": "พีชคณิตเชิงเส้น", "7": "การสุ่ม", "8": "ประกอบร่าง"}
    parts = {}
    for k in MAXPTS:
        s = k.split(".")[0]
        got, mx = parts.get(s, (0, 0))
        parts[s] = (got + SCORES.get(k, 0), mx + MAXPTS[k])
    print("%-4s %-34s %s" % ("", "", "score"))
    print("-" * 52)
    for s in sorted(parts, key=lambda z: int(z)):
        g, m = parts[s]
        print("%-4s %-34s %4d / %-4d" % (s, names.get(s, ""), g, m))
    print("-" * 52)
    print("%-39s %4d / %-4d" % ("TOTAL", sum(SCORES.values()), sum(MAXPTS.values())))


print("grade() and report() are ready.")

NumPy 2.4.6
grade() and report() are ready.


### Reference card / การ์ดอ้างอิง

ทุกอย่างที่แลปนี้สอน รวมไว้ที่เดียว เรียงตามหมวด เวลาติดในแลปถัดไปให้กลับมาดูที่นี่ก่อน
ช่องขวาสุดบอกว่าเคยฝึกในข้อไหน ย้อนกลับไปดูตัวอย่างที่คำนวณไว้แล้วได้

**1. รูปร่างและการหยิบข้อมูล**

| ทำอะไร | เขียนอย่างไร | ข้อ |
| --- | --- | --- |
| รูปร่าง / จำนวนมิติ | `A.shape`, `A.ndim` | |
| ทำให้เป็นคอลัมน์ | `a.reshape(-1, 1)` | 1.1 |
| สลับแถวกับคอลัมน์ | `A.T` | 1.2 |
| แถวที่ i / คอลัมน์ที่ j / ช่องเดียว | `A[i]` / `A[:, j]` / `A[i, j]` | 1.3 |
| เอาแถวเว้นแถว | `A[::2]` และ `A[1::2]` | 1.4 |
| ตัดหน้าต่าง s คูณ s | `A[i:i+s, j:j+s]` | 1.5 |
| แบนให้เหลือสองมิติ | `A.reshape(A.shape[0], -1)` | 1.6 |
| สร้างตารางศูนย์ / หนึ่ง | `np.zeros((n, m))`, `np.ones((n, m))` | |
| สร้างให้ขนาดเท่าของเดิม | `np.zeros_like(a)`, `np.ones_like(a)` | |

**2. คำนวณทีละตัว และ broadcasting**

| ทำอะไร | เขียนอย่างไร | ข้อ |
| --- | --- | --- |
| บวกลบคูณหารยกกำลัง ทีละตัว | `a + b`, `a * b`, `a ** 2` | 2.1 |
| เลข 0 ถึง p-1 | `np.arange(p)` | 2.2 |
| ต่อข้าง / ต่อล่าง | `np.concatenate([A, B], axis=1)` / `axis=0` | 2.3, 2.7 |
| ลบค่าเฉลี่ยรายคอลัมน์ | `X - X.mean(axis=0)` | 2.4 |
| เลขยกกำลัง e | `np.exp(z)` | 2.5 |
| หนีบค่าไม่ให้ล้น | `np.clip(z, -500, 500)` | 2.5 |
| softmax ที่ไม่ล้น | `np.exp(z - z.max())` แล้วหารด้วยผลรวม | 2.6 |

**3. รวมค่าและหาค่าสุดขีด**

| ทำอะไร | เขียนอย่างไร | ข้อ |
| --- | --- | --- |
| รวมทุกตัว / รวมตามแกน | `a.sum()` / `A.sum(axis=0)` | 3.1 |
| ค่าเฉลี่ยรายคอลัมน์ / รายแถว | `A.mean(axis=0)` / `A.mean(axis=1)` | 3.1 |
| รากที่สอง / log ฐาน 2 / log ธรรมชาติ | `np.sqrt(a)` / `np.log2(a)` / `np.log(a)` | 3.2, 3.3 |
| ระยะทางยุคลิด | `np.sqrt(((a-b)**2).sum())` หรือ `np.linalg.norm(a-b)` | 3.2 |
| ค่ามากสุด / ตำแหน่ง ตามแกน | `A.max(axis=1)` / `A.argmax(axis=1)` | 3.4 |

**4. boolean mask**

| ทำอะไร | เขียนอย่างไร | ข้อ |
| --- | --- | --- |
| เปรียบเทียบได้ boolean | `a > 3`, `a == v` | 4.1 |
| นับจำนวนที่จริง / สัดส่วนที่จริง | `(a > 3).sum()` / `(a > 3).mean()` | 4.1, 4.4 |
| เลือกด้วย mask | `a[mask]`, `A[:, mask]`, `~mask` | 4.2, 4.3 |
| ต่อหลายเงื่อนไข | `(a > 1) & (a < 5)` ใช้ `&` `|` ไม่ใช่ `and` `or` | |
| ความถูกต้องของการทำนาย | `(y == yhat).mean()` | 4.4 |
| one-hot | `np.eye(c)[y]` | 4.5 |

**5. unique, argmax, argsort**

| ทำอะไร | เขียนอย่างไร | ข้อ |
| --- | --- | --- |
| ค่าที่ไม่ซ้ำ | `np.unique(a)` | 5.1 |
| ค่าที่ไม่ซ้ำพร้อมจำนวน | `np.unique(a, return_counts=True)` | 5.2 |
| ตำแหน่งของค่ามากสุด | `np.argmax(a)` | 5.3 |
| ลำดับหลังเรียง / เอา k ตัวแรก | `np.argsort(a)` / `np.argsort(a)[:k]` | 5.4 |

**6. พีชคณิตเชิงเส้น**

| ทำอะไร | เขียนอย่างไร | ข้อ |
| --- | --- | --- |
| คูณเมทริกซ์ | `A @ B` ไม่ใช่ `A * B` | 6.1 |
| X ทรานสโพส คูณ X | `X.T @ X` | 6.2 |
| เมทริกซ์เอกลักษณ์ | `np.eye(p)` | 6.3 |
| แก้ระบบสมการ A z = b | `np.linalg.solve(A, b)` | 6.4 |
| เมทริกซ์ทแยง / outer product | `np.diag(v)` / `a @ a.T` เมื่อ a เป็น (c,1) | 6.5 |

**7. การสุ่ม**

| ทำอะไร | เขียนอย่างไร | ข้อ |
| --- | --- | --- |
| ตรึงผลให้รันซ้ำได้เหมือนเดิม | `np.random.seed(0)` | 7 |
| สุ่มตำแหน่งแบบคืนที่ | `np.random.randint(n, size=k)` | 7.1 |
| สุ่มตำแหน่งแบบไม่คืนที่ | `np.random.choice(p, m, replace=False)` | 7.2 |
| ปัดลง และรากที่สองของตัวเลข | `int(np.floor(np.sqrt(p)))` | 7.2 |
| สุ่มจำนวนจริง 0 ถึง 1 | `np.random.rand()` | 7.3 |

---
# 1. สร้าง array และมองรูปร่าง (16 คะแนน)

ความผิดพลาดที่เจอบ่อยที่สุดใน NumPy ไม่ใช่สูตรผิด แต่เป็น**รูปร่างผิด** และที่แย่คือมันมักไม่ error
มันคำนวณให้เฉย ๆ แต่ได้คำตอบที่ไม่ใช่สิ่งที่ตั้งใจ ดังนั้นสิ่งแรกที่ต้องติดเป็นนิสัยคือ
**พิมพ์ `.shape` ดูทุกครั้งที่สงสัย**

### ตัวอย่างที่ต้องแยกให้ออก

```python
a = np.array([1., 2., 3.])                 # shape (3,)    1 มิติ  ไม่ใช่แถว ไม่ใช่คอลัมน์
b = np.array([[1., 2., 3.]])               # shape (1, 3)  2 มิติ  หนึ่งแถว สามคอลัมน์
c = np.array([[1.], [2.], [3.]])           # shape (3, 1)  2 มิติ  สามแถว หนึ่งคอลัมน์
```

ทั้งสามอันมีเลข 1, 2, 3 เหมือนกันหมด แต่ NumPy ปฏิบัติกับมันคนละแบบ

$$a:\; \begin{bmatrix}1 & 2 & 3\end{bmatrix}_{(3,)}
\qquad b:\; \begin{bmatrix}1 & 2 & 3\end{bmatrix}_{(1,3)}
\qquad c:\; \begin{bmatrix}1\\2\\3\end{bmatrix}_{(3,1)}$$

รันเซลล์ข้างล่างนี้ดูด้วยตาตัวเองก่อน

In [2]:
a = np.array([1., 2., 3.])
b = np.array([[1., 2., 3.]])
c = np.array([[1.], [2.], [3.]])

for name, arr in [("a", a), ("b", b), ("c", c)]:
    print("%s  shape %-8s ndim %d" % (name, str(arr.shape), arr.ndim))
    print(arr, "\n")

a  shape (3,)     ndim 1
[1. 2. 3.] 

b  shape (1, 3)   ndim 2
[[1. 2. 3.]] 

c  shape (3, 1)   ndim 2
[[1.]
 [2.]
 [3.]] 



### 1.1 `as_column` (4 คะแนน)

ทำ array 1 มิติ ให้กลายเป็นคอลัมน์

**ตัวอย่างที่คำนวณให้แล้ว**

```
ใส่เข้าไป  np.array([2., 3., 4., 5.])        shape (4,)
ได้ออกมา   [[2.],
            [3.],
            [4.],
            [5.]]                            shape (4, 1)
```

`-1` แปลว่า "คิดให้เอง" มีสมาชิก 4 ตัว บังคับ 1 คอลัมน์ ก็ต้องเป็น 4 แถว

**คำใบ้** `x.reshape(-1, 1)`

In [3]:
def as_column(x):
    '''
    เปลี่ยน array 1 มิติ ให้เป็นคอลัมน์ 2 มิติ
    ใส่เข้า : x ขนาด (n,)
    ได้ออก  : out ขนาด (n, 1)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = x.reshape(-1, 1)
    #########################################
    return out

In [4]:
def test_as_column():
    out = as_column(np.array([2., 3., 4., 5.]))
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (4, 1), "ต้องได้รูปร่าง (4, 1) แต่ได้ %s" % (out.shape,)
    assert np.allclose(out, [[2.], [3.], [4.], [5.]])
    assert as_column(np.arange(7.)).shape == (7, 1), "ต้องใช้ได้กับความยาวอื่นด้วย"


grade("1.1 as_column", 4, test_as_column)

[PASS] 1.1 as_column                      4 / 4


### 1.2 `transpose` (4 คะแนน)

สลับแถวกับคอลัมน์

**ตัวอย่างที่คำนวณให้แล้ว**

$$A = \begin{bmatrix}1 & 2 & 3\\ 4 & 5 & 6\end{bmatrix}_{(2,3)}
\qquad\Longrightarrow\qquad
A^\top = \begin{bmatrix}1 & 4\\ 2 & 5\\ 3 & 6\end{bmatrix}_{(3,2)}$$

```
A[0, 1] คือ 2        หลัง transpose ไปอยู่ที่ A.T[1, 0]
A[1, 2] คือ 6        หลัง transpose ไปอยู่ที่ A.T[2, 1]
```

**คำใบ้** `A.T`

In [5]:
def transpose(A):
    '''
    สลับแถวกับคอลัมน์
    ใส่เข้า : A ขนาด (n, p)
    ได้ออก  : out ขนาด (p, n)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = A.T
    #########################################
    return out

In [6]:
def test_transpose():
    A = np.array([[1., 2., 3.],
                  [4., 5., 6.]])
    out = transpose(A)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (3, 2), "ต้องได้ (3, 2) แต่ได้ %s" % (out.shape,)
    assert np.allclose(out, [[1., 4.], [2., 5.], [3., 6.]])
    assert out[1, 0] == 2., "A[0,1] ต้องย้ายไปอยู่ที่ out[1,0]"


grade("1.2 transpose", 4, test_transpose)

[PASS] 1.2 transpose                      4 / 4


### การอ้างถึงแถวและคอลัมน์

เครื่องหมายจุลภาคในวงเล็บเหลี่ยมคือการบอกว่า "แกนที่หนึ่ง, แกนที่สอง" และ `:` แปลว่า "เอาหมด"

$$X = \begin{bmatrix}
10 & 11 & 12\\
20 & 21 & 22\\
30 & 31 & 32\\
40 & 41 & 42
\end{bmatrix}_{(4,3)}$$

| เขียน | ได้ | รูปร่าง | ความหมาย |
| --- | --- | --- | --- |
| `X[1]` | `[20, 21, 22]` | `(3,)` | แถวที่ 1 ทั้งแถว |
| `X[1, :]` | `[20, 21, 22]` | `(3,)` | เหมือนกันเป๊ะ เขียนให้ชัดขึ้น |
| `X[:, 2]` | `[12, 22, 32, 42]` | `(4,)` | คอลัมน์ที่ 2 ทั้งคอลัมน์ |
| `X[1, 2]` | `22` | ตัวเลขเดียว | แถว 1 คอลัมน์ 2 |
| `X[1:3, :]` | สองแถวกลาง | `(2,3)` | แถว 1 ถึง 2 (ไม่รวม 3) |

**สังเกตว่า `X[:, 2]` ได้รูปร่าง `(4,)` ไม่ใช่ `(4, 1)`** การตัดคอลัมน์ออกมาทำให้มิติหายไปหนึ่งมิติ
ถ้าต้องการให้เป็นคอลัมน์จริง ๆ ต้อง `X[:, 2].reshape(-1, 1)` หรือ `X[:, 2:3]`

In [7]:
X = np.array([[10., 11., 12.],
              [20., 21., 22.],
              [30., 31., 32.],
              [40., 41., 42.]])

print("X[1]       =", X[1], "  shape", X[1].shape)
print("X[:, 2]    =", X[:, 2], "  shape", X[:, 2].shape)
print("X[1, 2]    =", X[1, 2])
print("X[:, 2:3]  shape", X[:, 2:3].shape, "  <- ยังเป็นคอลัมน์อยู่")

X[1]       = [20. 21. 22.]   shape (3,)
X[:, 2]    = [12. 22. 32. 42.]   shape (4,)
X[1, 2]    = 22.0
X[:, 2:3]  shape (4, 1)   <- ยังเป็นคอลัมน์อยู่


### 1.3 `row_and_col` (4 คะแนน)

คืนแถวที่ `i` และคอลัมน์ที่ `j` ออกมาเป็นคู่

**ตัวอย่างที่คำนวณให้แล้ว** ใช้ `X` ข้างบน กับ `i = 2`, `j = 0`

```
row = X[2]      = [30., 31., 32.]
col = X[:, 0]   = [10., 20., 30., 40.]
```

**คำใบ้** `A[i]` กับ `A[:, j]` คืนค่าทั้งคู่ด้วย `return row, col`

In [8]:
def row_and_col(A, i, j):
    '''
    หยิบแถวที่ i และคอลัมน์ที่ j
    ใส่เข้า : A ขนาด (n, p), i และ j เป็นจำนวนเต็ม
    ได้ออก  : row ขนาด (p,) และ col ขนาด (n,)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    row, col = A[i], A[:, j]
    #########################################
    return row, col

In [9]:
def test_row_and_col():
    A = np.array([[10., 11., 12.],
                  [20., 21., 22.],
                  [30., 31., 32.],
                  [40., 41., 42.]])
    row, col = row_and_col(A, 2, 0)
    assert row is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert row.shape == (3,), "แถวต้องยาว 3 แต่ได้ %s  (สลับ i กับ j หรือเปล่า)" % (row.shape,)
    assert col.shape == (4,), "คอลัมน์ต้องยาว 4 แต่ได้ %s" % (col.shape,)
    assert np.allclose(row, [30., 31., 32.])
    assert np.allclose(col, [10., 20., 30., 40.])


grade("1.3 row_and_col", 4, test_row_and_col)

[PASS] 1.3 row_and_col                    4 / 4


### 1.4 `split_half` (4 คะแนน)

แบ่งข้อมูลเป็นชุดฝึกกับชุดทดสอบ ด้วยการเอาแถวเว้นแถว

**ตัวอย่างที่คำนวณให้แล้ว** ข้อมูล 6 แถว

```
X        = [ r0, r1, r2, r3, r4, r5 ]
X[::2]   = [ r0,     r2,     r4     ]      เริ่มที่ 0 ก้าวทีละ 2
X[1::2]  = [     r1,     r3,     r5 ]      เริ่มที่ 1 ก้าวทีละ 2
```

ไวยากรณ์คือ `[เริ่ม : จบ : ก้าว]` ถ้าเว้นว่างไว้แปลว่าตั้งแต่ต้นจนจบ

**คำใบ้** `A[::2]` และ `A[1::2]`

In [10]:
def split_half(A):
    '''
    แบ่งแถวเว้นแถว
    ใส่เข้า : A ขนาด (n, p)
    ได้ออก  : train คือแถวที่ 0, 2, 4, ... และ test คือแถวที่ 1, 3, 5, ...
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    train, test = A[::2], A[1::2]
    #########################################
    return train, test

In [11]:
def test_split_half():
    A = np.arange(12.).reshape(6, 2)
    train, test = split_half(A)
    assert train is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert train.shape == (3, 2) and test.shape == (3, 2)
    assert np.allclose(train[:, 0], [0., 4., 8.]), "train ต้องเป็นแถวที่ 0, 2, 4"
    assert np.allclose(test[:, 0], [2., 6., 10.]), "test ต้องเป็นแถวที่ 1, 3, 5"

    a = np.arange(7.)
    tr, te = split_half(a)
    assert len(tr) == 4 and len(te) == 3, "ถ้าจำนวนแถวเป็นเลขคี่ ชุดแรกจะได้มากกว่าหนึ่งแถว"


grade("1.4 split_half", 4, test_split_half)

[PASS] 1.4 split_half                     4 / 4


### 1.5 `window` (4 คะแนน)

ตัดสี่เหลี่ยมจัตุรัสขนาด `s` คูณ `s` ออกมาจากตาราง โดยเริ่มที่มุมบนซ้ายตำแหน่ง `(i, j)`

**ตัวอย่างที่คำนวณให้แล้ว**

$$A = \begin{bmatrix}
1 & 2 & 3 & 4\\
5 & \mathbf{6} & \mathbf{7} & 8\\
9 & \mathbf{10} & \mathbf{11} & 12\\
13 & 14 & 15 & 16\end{bmatrix}
\qquad i = 1,\; j = 1,\; s = 2$$

```
A[1:3, 1:3] = [[ 6,  7],
               [10, 11]]
```

อ่านว่า เอาแถว 1 ถึง 2 (ไม่รวม 3) และคอลัมน์ 1 ถึง 2 (ไม่รวม 3)
ใช้จุลภาคคั่นสองแกน เหมือนข้อ 1.3 แต่คราวนี้ใส่ช่วงทั้งสองฝั่ง

**คำใบ้** `A[i:i+s, j:j+s]`

In [12]:
def window(A, i, j, s):
    '''
    ตัดหน้าต่างขนาด s คูณ s ออกมาจาก A โดยมุมบนซ้ายอยู่ที่ (i, j)
    ใส่เข้า : A ขนาด (h, w), i และ j เป็นจำนวนเต็ม, s คือขนาดของหน้าต่าง
    ได้ออก  : out ขนาด (s, s)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    out = A[i:i+s, j:j+s]
    #########################################
    return out

In [13]:
def test_window():
    A = np.arange(1., 17.).reshape(4, 4)
    out = window(A, 1, 1, 2)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (2, 2), "ต้องได้ (2, 2) แต่ได้ %s" % (out.shape,)
    assert np.allclose(out, [[6., 7.], [10., 11.]])
    assert np.allclose(window(A, 0, 0, 3), [[1., 2., 3.], [5., 6., 7.], [9., 10., 11.]])


def test_window_is_a_convolution_step():
    '''หนึ่งก้าวของ convolution คือหยิบหน้าต่าง คูณกับตัวกรอง แล้วบวกกันหมด'''
    A = np.arange(1., 17.).reshape(4, 4)
    W = np.array([[1., 0.], [0., 1.]])
    assert np.allclose((window(A, 1, 1, 2) * W).sum(), 6. + 11.)


grade("1.5 window", 4, test_window, test_window_is_a_convolution_step)

[PASS] 1.5 window                         4 / 4


### 1.6 `flatten` (4 คะแนน)

ยุบทุกมิติให้เหลือสองมิติ โดยเก็บมิติแรกไว้ และรวมมิติที่เหลือทั้งหมดเป็นมิติเดียว

**ตัวอย่างที่คำนวณให้แล้ว** ผลลัพธ์จากชั้น convolution ของรูป 2 ใบ

```
A ขนาด (2, 3, 4, 4)
     2 = จำนวนรูปในแบตช์     <- มิตินี้เก็บไว้
     3 = จำนวนตัวกรอง
     4 = ความสูง
     4 = ความกว้าง

3 x 4 x 4 = 48

ได้ออกมา (2, 48)   คือรูปละหนึ่งแถว แถวละ 48 ตัวเลข
```

`-1` ยังแปลว่า คิดให้เอง เหมือนข้อ 1.1 แต่คราวนี้ล็อกมิติแรกไว้แทนที่จะล็อกคอลัมน์

**ทำไมต้องทำ** ชั้น fully connected รับข้อมูลเป็นตารางสองมิติเท่านั้น คือ หนึ่งแถวต่อหนึ่งตัวอย่าง
แต่ชั้น convolution คายผลออกมาเป็นสี่มิติ จึงต้องแบนก่อนส่งต่อ

ใน PyTorch คำสั่งคือ `.view(-1, n)` ซึ่งทำสิ่งเดียวกัน

**คำใบ้** `A.reshape(A.shape[0], -1)`

In [14]:
def flatten(A):
    '''
    ยุบทุกมิติยกเว้นมิติแรก
    ใส่เข้า : A มีกี่มิติก็ได้ เช่น (n, f, h, w)
    ได้ออก  : out ขนาด (n, f*h*w)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    out = A.reshape(A.shape[0], -1)
    #########################################
    return out

In [15]:
def test_flatten():
    A = np.arange(96.).reshape(2, 3, 4, 4)
    out = flatten(A)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (2, 48), "ต้องได้ (2, 48) แต่ได้ %s" % (out.shape,)
    assert np.allclose(out[0], np.arange(48.)), "ลำดับตัวเลขต้องไม่เปลี่ยน"
    assert np.allclose(out[1], np.arange(48., 96.))


def test_flatten_other_shapes():
    assert flatten(np.zeros((5, 2, 7))).shape == (5, 14)
    assert flatten(np.zeros((5, 3))).shape == (5, 3), \
        "ถ้าเป็นสองมิติอยู่แล้ว ต้องไม่เปลี่ยนอะไร"
    assert flatten(np.zeros((1, 10, 10))).shape == (1, 100)


grade("1.6 flatten", 4, test_flatten, test_flatten_other_shapes)

[PASS] 1.6 flatten                        4 / 4


---
# 2. คำนวณทีละตัว และ broadcasting (16 คะแนน)

`+`, `-`, `*`, `/`, `**` ใน NumPy ทำงาน **ทีละตัวต่อตัว** ไม่ใช่การคูณเมทริกซ์

$$\begin{bmatrix}1\\2\\3\end{bmatrix} \ast \begin{bmatrix}10\\20\\30\end{bmatrix}
= \begin{bmatrix}10\\40\\90\end{bmatrix}
\qquad\text{ไม่ใช่ inner product}$$

### broadcasting คืออะไร

ถ้ารูปร่างไม่เท่ากัน NumPy จะ **ทำซ้ำฝั่งที่ขนาดเป็น 1 ให้เท่าอีกฝั่ง** แล้วค่อยคำนวณทีละตัว

**ตัวอย่างที่คำนวณให้แล้ว** คอลัมน์ `(2,1)` ยกกำลัง แถว `(3,)`

$$\begin{bmatrix}2\\3\end{bmatrix}_{(2,1)}
\text{ ยกกำลัง }
\begin{bmatrix}0&1&2\end{bmatrix}_{(3,)}$$

$$\begin{bmatrix}2^0 & 2^1 & 2^2\\ 3^0 & 3^1 & 3^2\end{bmatrix}
= \begin{bmatrix}1 & 2 & 4\\ 1 & 3 & 9\end{bmatrix}_{(2,3)}$$

```
x = [[2],      ขนาด (2,1)        e = [0, 1, 2]     ขนาด (3,)
     [3]]

ทำซ้ำไปทางขวา 3 ครั้ง            ทำซ้ำลงมา 2 ครั้ง
    [[2, 2, 2],                      [[0, 1, 2],
     [3, 3, 3]]                       [0, 1, 2]]

ยกกำลังตรงตำแหน่ง
    [[2^0, 2^1, 2^2],     =     [[1, 2, 4],
     [3^0, 3^1, 3^2]]            [1, 3, 9]]        ขนาด (2,3)
```

**กฎข้อเดียว** ขนาดในแต่ละตำแหน่งต้อง **เท่ากัน** หรือ **ฝั่งใดฝั่งหนึ่งเป็น 1**
ถ้าไม่เข้าทั้งสองกรณี NumPy จะ error

```
(3,) กับ (4,)  ->  ValueError: operands could not be broadcast together
```

**ปัญหาที่จะเจอ** `(3,1)` กับ `(3,)` ได้ตาราง `(3,3)` เก้าช่อง ไม่ใช่สามช่อง และ **ไม่มี error**
เพราะเข้ากฎข้างบน ถ้างานของคุณได้ตัวเลขเยอะผิดปกติ ให้สงสัยข้อนี้ก่อน

In [16]:
# ทายก่อนรัน: สองบรรทัดนี้จะได้รูปร่างอะไร
# แก้สองบรรทัดล่างเป็นคำตอบที่คุณทาย เช่น  guess1 = "(3,)"
guess1 = "(3,)"
guess2 = "(3, 3)"

a = np.array([1., 2., 3.])          # (3,)
c = np.array([[1.], [2.], [3.]])    # (3, 1)

print("คุณทาย shape ของ a * a ได้", guess1, " คำตอบจริง", (a * a).shape)
print("คุณทาย shape ของ c * a ได้", guess2, " คำตอบจริง", (c * a).shape)
print()
print("a * a =", a * a)
print("c * a =")
print(c * a)
print()
print("บรรทัดที่สองคือปัญหาที่เจอบ่อยที่สุด (3,1) เจอกับ (3,) กลายเป็นตาราง 3x3")
print("ถ้าเห็นตัวเลขเยอะผิดปกติในงานของตัวเอง ให้สงสัยข้อนี้ก่อน")

คุณทาย shape ของ a * a ได้ (3,)  คำตอบจริง (3,)
คุณทาย shape ของ c * a ได้ (3, 3)  คำตอบจริง (3, 3)

a * a = [1. 4. 9.]
c * a =
[[1. 2. 3.]
 [2. 4. 6.]
 [3. 6. 9.]]

บรรทัดที่สองคือปัญหาที่เจอบ่อยที่สุด (3,1) เจอกับ (3,) กลายเป็นตาราง 3x3
ถ้าเห็นตัวเลขเยอะผิดปกติในงานของตัวเอง ให้สงสัยข้อนี้ก่อน


### 2.1 `squared_error` (4 คะแนน)

ผลต่างยกกำลังสอง ทีละตัว

**ตัวอย่างที่คำนวณให้แล้ว**

```
y     = [3., 5., 2.]
yhat  = [2., 5., 4.]
ผลต่าง  = [1., 0., -2.]
ยกกำลังสอง = [1., 0., 4.]
```

$$(y_i - \hat{y}_i)^2 \;=\; (3-2)^2,\; (5-5)^2,\; (2-4)^2 \;=\; 1,\; 0,\; 4$$

**คำใบ้** `(y - yhat) ** 2` บรรทัดเดียว ไม่ต้องวนลูป

In [17]:
def squared_error(y, yhat):
    '''
    ผลต่างยกกำลังสอง ทีละตัว
    ใส่เข้า : y และ yhat ขนาดเท่ากัน
    ได้ออก  : out ขนาดเท่าเดิม แต่ละตัวคือ (y[i] - yhat[i]) ** 2
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = (y - yhat) ** 2
    #########################################
    return out

In [18]:
def test_squared_error():
    out = squared_error(np.array([3., 5., 2.]), np.array([2., 5., 4.]))
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (3,), "ต้องได้ขนาดเท่าเดิม ไม่ใช่ตัวเลขตัวเดียว (ยังไม่ต้องรวม)"
    assert np.allclose(out, [1., 0., 4.])
    assert np.allclose(squared_error(np.zeros(4), np.array([1., -1., 2., -2.])),
                       [1., 1., 4., 4.]), "ยกกำลังสองแล้วต้องไม่มีค่าติดลบ"


grade("2.1 squared_error", 4, test_squared_error)

[PASS] 2.1 squared_error                  4 / 4


### 2.2 `powers` (4 คะแนน)

สร้างตารางยกกำลัง นี่คือ `compute_Phi` ของ Lab 1 เป๊ะ ๆ

**ตัวอย่างที่คำนวณให้แล้ว** `x = [2., 3., 4.]` และ `p = 4`

$$\begin{bmatrix}2\\3\\4\end{bmatrix} \text{ กับเลขชี้กำลัง } \begin{bmatrix}0&1&2&3\end{bmatrix}
\;\Longrightarrow\;
\begin{bmatrix}
2^0 & 2^1 & 2^2 & 2^3\\
3^0 & 3^1 & 3^2 & 3^3\\
4^0 & 4^1 & 4^2 & 4^3
\end{bmatrix}
= \begin{bmatrix}
1 & 2 & 4 & 8\\
1 & 3 & 9 & 27\\
1 & 4 & 16 & 64
\end{bmatrix}_{(3,4)}$$

สามขั้น

1. `np.arange(4)` ได้ `[0, 1, 2, 3]` รูปร่าง `(4,)`
2. `x.reshape(-1, 1)` ได้รูปร่าง `(3, 1)` นี่คือข้อ 1.1
3. `(3, 1) ** (4,)` broadcasting ออกมาเป็น `(3, 4)`

**คำใบ้** รวมสามขั้นเป็นบรรทัดเดียวได้ ลองเขียนแยกสามบรรทัดก่อนแล้วค่อยรวมก็ได้

In [19]:
def powers(x, p):
    '''
    ตารางยกกำลังของ x
    ใส่เข้า : x ขนาด (n,) หรือ (n, 1), p เป็นจำนวนเต็ม
    ได้ออก  : out ขนาด (n, p) โดย out[i, j] = x[i] ** j
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = x.reshape(-1, 1) ** np.arange(p)
    #########################################
    return out

In [20]:
def test_powers():
    out = powers(np.array([2., 3., 4.]), 4)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (3, 4), "ต้องได้ (3, 4) แต่ได้ %s  (ลืม reshape(-1,1) หรือเปล่า)" % (out.shape,)
    assert np.allclose(out, [[1., 2., 4., 8.],
                             [1., 3., 9., 27.],
                             [1., 4., 16., 64.]])


def test_powers_other_sizes():
    out = powers(np.array([5.]), 3)
    assert out.shape == (1, 3) and np.allclose(out, [[1., 5., 25.]])
    assert np.allclose(powers(np.array([1., 2.]), 1), [[1.], [1.]]), \
        "p=1 แปลว่ามีแต่คอลัมน์ x^0 ซึ่งเป็น 1 ทั้งหมด"


grade("2.2 powers", 4, test_powers, test_powers_other_sizes)

[PASS] 2.2 powers                         4 / 4


### 2.3 `add_bias_column` (4 คะแนน)

เติมคอลัมน์ของเลข 1 ไว้หน้าสุด เพื่อให้ค่าคงที่ $b$ ในโมเดลกลายเป็นน้ำหนักตัวหนึ่งใน $\mathbf{w}$

**ตัวอย่างที่คำนวณให้แล้ว**

$$X = \begin{bmatrix}7 & 8\\ 9 & 10\\ 11 & 12\end{bmatrix}_{(3,2)}
\;\Longrightarrow\;
\begin{bmatrix}1 & 7 & 8\\ 1 & 9 & 10\\ 1 & 11 & 12\end{bmatrix}_{(3,3)}$$

ทำไมถึงช่วย เพราะ $w_1 x_1 + w_2 x_2 + b$ เขียนใหม่ได้เป็น
$w_0 \cdot 1 + w_1 x_1 + w_2 x_2$ แปลว่าไม่ต้องจัดการ $b$ แยกอีกต่อไป

**คำใบ้**
- `np.ones((n, 1))` สร้างคอลัมน์ของเลข 1 สูง n แถว
- `np.hstack([A, B])` เอามาต่อกันในแนวนอน ใส่เป็น list นะครับ มีวงเล็บเหลี่ยม
- `n` หาได้จาก `X.shape[0]`

In [21]:
def add_bias_column(X):
    '''
    เติมคอลัมน์ของเลข 1 ไว้หน้าสุด
    ใส่เข้า : X ขนาด (n, p)
    ได้ออก  : out ขนาด (n, p + 1) โดยคอลัมน์แรกเป็น 1 ทั้งหมด
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = np.hstack([np.ones((X.shape[0], 1)), X])
    #########################################
    return out

In [22]:
def test_add_bias_column():
    X = np.array([[7., 8.],
                  [9., 10.],
                  [11., 12.]])
    out = add_bias_column(X)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (3, 3), "ต้องได้ (3, 3) แต่ได้ %s" % (out.shape,)
    assert np.allclose(out[:, 0], 1.), "คอลัมน์แรกต้องเป็น 1 ทั้งหมด"
    assert np.allclose(out[:, 1:], X), "คอลัมน์ที่เหลือต้องเป็นข้อมูลเดิม ไม่เปลี่ยนลำดับ"


def test_add_bias_column_one_feature():
    out = add_bias_column(np.array([[5.], [6.]]))
    assert out.shape == (2, 2)
    assert np.allclose(out, [[1., 5.], [1., 6.]])


grade("2.3 add_bias_column", 4, test_add_bias_column, test_add_bias_column_one_feature)

[PASS] 2.3 add_bias_column                4 / 4


### 2.4 `center` (4 คะแนน)

ลบค่าเฉลี่ยของแต่ละคอลัมน์ออก ให้ทุกคอลัมน์มีค่าเฉลี่ยเป็นศูนย์

**ตัวอย่างที่คำนวณให้แล้ว**

$$X = \begin{bmatrix}1 & 10\\ 3 & 20\\ 5 & 30\end{bmatrix}
\qquad
\text{ค่าเฉลี่ยแต่ละคอลัมน์} = \begin{bmatrix}3 & 20\end{bmatrix}_{(2,)}$$

$$X - \begin{bmatrix}3 & 20\end{bmatrix}
= \begin{bmatrix}1-3 & 10-20\\ 3-3 & 20-20\\ 5-3 & 30-20\end{bmatrix}
= \begin{bmatrix}-2 & -10\\ 0 & 0\\ 2 & 10\end{bmatrix}$$

สังเกตว่า `(3, 2)` ลบกับ `(2,)` ได้เลย เพราะ broadcasting ยืดแถวเดียวนั้นให้ครบทั้งสามแถว
**นี่คือเหตุผลที่ต้องใช้ `axis=0` ไม่ใช่ `axis=1`** ถ้าใช้ `axis=1` จะได้ค่าเฉลี่ยของแต่ละแถว
ซึ่งเป็นคนละเรื่อง

**คำใบ้** `X - X.mean(axis=0)`

In [23]:
def center(X):
    '''
    ลบค่าเฉลี่ยของแต่ละคอลัมน์ออก
    ใส่เข้า : X ขนาด (n, p)
    ได้ออก  : out ขนาด (n, p) ที่ทุกคอลัมน์มีค่าเฉลี่ยเป็น 0
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = X - X.mean(axis=0)
    #########################################
    return out

In [24]:
def test_center():
    X = np.array([[1., 10.],
                  [3., 20.],
                  [5., 30.]])
    out = center(X)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (3, 2), "ต้องได้ขนาดเท่าเดิม แต่ได้ %s" % (out.shape,)
    assert np.allclose(out, [[-2., -10.], [0., 0.], [2., 10.]])


def test_center_means_are_zero():
    rng = np.random.default_rng(0)
    X = rng.normal(5., 3., (20, 4))
    out = center(X)
    assert np.allclose(out.mean(axis=0), 0., atol=1e-10), \
        "ค่าเฉลี่ยของแต่ละคอลัมน์ต้องเป็น 0  (ใช้ axis=0 หรือ axis=1)"


grade("2.4 center", 4, test_center, test_center_means_are_zero)

[PASS] 2.4 center                         4 / 4


### 2.5 `sigmoid` (4 คะแนน)

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**ตัวอย่างที่คำนวณให้แล้ว**

```
z = 0     ->  1/(1 + e^0)    = 1/(1+1)      = 0.5
z = 1     ->  1/(1 + e^-1)   = 1/(1+0.3679) = 0.7311
z = -1    ->  1/(1 + e^1)    = 1/(1+2.7183) = 0.2689
```

**ปัญหาที่จะเจอแน่ ๆ** ถ้า z ติดลบมาก ๆ เช่น z = -1000 จะต้องคำนวณ $e^{1000}$
ซึ่งใหญ่เกินกว่าที่คอมพิวเตอร์เก็บได้ NumPy จะเตือน `RuntimeWarning: overflow in exp`
วิธีแก้คือ **หนีบค่า z ไว้ก่อน** ด้วย `np.clip(z, -500, 500)` เพราะ $e^{500}$ ยังเก็บได้
และ $\sigma(-500)$ กับ $\sigma(-1000)$ ก็เท่ากับ 0 เหมือนกันอยู่ดี

**คำใบ้** `z = np.clip(z, -500, 500)` แล้วค่อย `1 / (1 + np.exp(-z))`

In [25]:
def sigmoid(z):
    '''
    ฟังก์ชันซิกมอยด์ ทำงานทีละตัว และต้องไม่ล้นเมื่อ z มีค่ามากหรือน้อยมาก
    ใส่เข้า : z ขนาดเท่าไหร่ก็ได้
    ได้ออก  : a ขนาดเท่าเดิม ทุกค่าอยู่ระหว่าง 0 ถึง 1
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    z = np.clip(z, -500, 500)
    a = 1 / (1 + np.exp(-z))
    #########################################
    return a

In [26]:
def test_sigmoid():
    a = sigmoid(np.array([0., 1., -1.]))
    assert a is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert np.allclose(a, [0.5, 0.7311, 0.2689], atol=1e-4)


def test_sigmoid_no_overflow():
    with np.errstate(over="raise"):
        a = sigmoid(np.array([-1000., -500., 0., 500., 1000.]))
    assert np.all(np.isfinite(a)), "ได้ค่า inf หรือ nan ลืม np.clip หรือเปล่า"
    assert a[0] < 1e-10 and a[-1] > 1 - 1e-10
    assert np.all((a >= 0) & (a <= 1)), "ซิกมอยด์ต้องอยู่ระหว่าง 0 ถึง 1 เสมอ"


grade("2.5 sigmoid", 4, test_sigmoid, test_sigmoid_no_overflow)

[PASS] 2.5 sigmoid                        4 / 4


### 2.6 `stable_softmax` (4 คะแนน)

$$a_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

แปลง logit เป็นความน่าจะเป็นที่รวมกันได้ 1

**ตัวอย่างที่คำนวณให้แล้ว** $z = [1, 2, 3]$

```
ลบค่ามากสุด (คือ 3) ออกก่อน  ->  [-2, -1, 0]

e^-2 = 0.1353
e^-1 = 0.3679
e^0  = 1.0000
รวม  = 1.5032

0.1353 / 1.5032 = 0.0900
0.3679 / 1.5032 = 0.2447
1.0000 / 1.5032 = 0.6652
```

**ทำไมต้องลบค่ามากสุด** ลองเปลี่ยนเป็น $z = [1000, 1001, 1002]$ ดู ถ้าคำนวณตรง ๆ จะต้องหา
$e^{1002}$ ซึ่งล้นทันที ได้ `inf` แล้วหาร `inf/inf` ออกมาเป็น `nan` แต่ถ้าลบ 1002 ออกก่อน
จะเหลือ $[-2, -1, 0]$ เท่าเดิม **และได้คำตอบชุดเดียวกันเป๊ะ** คือ $[0.0900, 0.2447, 0.6652]$

เหตุผลพิสูจน์ได้สองบรรทัด ให้ $m$ เป็นค่าใดก็ได้

$$\frac{e^{z_i - m}}{\sum_j e^{z_j - m}}
= \frac{e^{z_i} e^{-m}}{e^{-m} \sum_j e^{z_j}}
= \frac{e^{z_i}}{\sum_j e^{z_j}}$$

ตัว $e^{-m}$ อยู่ทั้งเศษและส่วน จึงตัดกันหมด เราเลือก $m$ เป็นค่ามากสุดเพราะทำให้
เลขชี้กำลังทุกตัวไม่เกินศูนย์ จึงไม่มีทางล้น

**ข้อสังเกตที่ควรจำ** softmax สนใจแค่**ผลต่าง**ระหว่าง logit ไม่ได้สนใจค่าสัมบูรณ์
$[1,2,3]$ กับ $[1000,1001,1002]$ ให้คำตอบเหมือนกัน

**คำใบ้** สามบรรทัด ลบ `z.max()` ออก แล้ว `np.exp` แล้วหารด้วยผลรวมของตัวเอง

In [27]:
def stable_softmax(z):
    '''
    แปลง logit เป็นความน่าจะเป็น โดยไม่ล้นแม้ z จะมีค่ามาก
    ใส่เข้า : z ขนาด (c,)
    ได้ออก  : a ขนาด (c,) ทุกค่าเป็นบวกและรวมกันได้ 1
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    z = z - z.max()
    e = np.exp(z)
    a = e / e.sum()
    #########################################
    return a

In [28]:
def test_stable_softmax():
    a = stable_softmax(np.array([1., 2., 3.]))
    assert a is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert np.allclose(a, [0.0900, 0.2447, 0.6652], atol=1e-4)
    assert np.allclose(a.sum(), 1.), "ความน่าจะเป็นต้องรวมกันได้ 1"
    assert np.allclose(stable_softmax(np.array([0., 0.])), [0.5, 0.5])


def test_stable_softmax_no_overflow():
    with np.errstate(over="raise", invalid="raise"):
        a = stable_softmax(np.array([1000., 1001., 1002.]))
    assert np.all(np.isfinite(a)), "ได้ inf หรือ nan ลืมลบค่ามากสุดออกก่อนหรือเปล่า"
    assert np.allclose(a, [0.0900, 0.2447, 0.6652], atol=1e-4), \
        "การเลื่อน z ทั้งชุดต้องไม่เปลี่ยนคำตอบ"


def test_stable_softmax_shift_invariant():
    z = np.array([-0.5, 2.1, 0.3, 1.0])
    for m in [0., 10., -37., 500.]:
        assert np.allclose(stable_softmax(z), stable_softmax(z + m), atol=1e-10), \
            "บวกค่าคงที่เข้าไปใน z ทุกตัว คำตอบต้องเท่าเดิม"


grade("2.6 stable_softmax", 4,
      test_stable_softmax, test_stable_softmax_no_overflow, test_stable_softmax_shift_invariant)

[PASS] 2.6 stable_softmax                 4 / 4


### 2.7 `concat_both` (4 คะแนน)

ต่อสองตารางเข้าด้วยกัน ทำได้สองทิศทาง

**ตัวอย่างที่คำนวณให้แล้ว**

$$A = \begin{bmatrix}1&2&3\\ 4&5&6\end{bmatrix}_{(2,3)}
\qquad
B = \begin{bmatrix}7&8&9\\ 10&11&12\end{bmatrix}_{(2,3)}$$

**ต่อข้าง** `axis=1` จำนวนคอลัมน์เพิ่ม จำนวนแถวเท่าเดิม

$$\begin{bmatrix}1&2&3&7&8&9\\ 4&5&6&10&11&12\end{bmatrix}_{(2,6)}$$

**ต่อล่าง** `axis=0` จำนวนแถวเพิ่ม จำนวนคอลัมน์เท่าเดิม

$$\begin{bmatrix}1&2&3\\ 4&5&6\\ 7&8&9\\ 10&11&12\end{bmatrix}_{(4,3)}$$

**กฎข้อเดียว** แกนที่ต่อคือแกนที่โตขึ้น **แกนที่เหลือต้องเท่ากันอยู่แล้ว**
ดังนั้น `(2,3)` กับ `(2,4)` ต่อได้เฉพาะ `axis=1` ส่วน `(2,3)` กับ `(5,3)` ต่อได้เฉพาะ `axis=0`

ใน PyTorch คำสั่งคือ `th.cat((x, H), 1)`

**คำใบ้** `np.concatenate([A, B], axis=1)` สังเกตว่าตารางสองก้อนใส่ใน list วงเล็บเหลี่ยม
ถ้าจำง่ายกว่า จะใช้ `np.hstack` กับ `np.vstack` ก็ได้ผลเดียวกันสำหรับสองมิติ

In [29]:
def concat_both(A, B):
    '''
    ต่อสองตารางทั้งสองทิศทาง
    ใส่เข้า : A และ B ขนาด (n, p) เท่ากัน
    ได้ออก  : side ขนาด (n, 2p) ต่อข้าง และ down ขนาด (2n, p) ต่อล่าง
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    side = np.concatenate([A, B], axis=1)
    down = np.concatenate([A, B], axis=0)
    #########################################
    return side, down

In [30]:
def test_concat_both():
    A = np.array([[1., 2., 3.],
                  [4., 5., 6.]])
    B = np.array([[7., 8., 9.],
                  [10., 11., 12.]])
    side, down = concat_both(A, B)
    assert side is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert side.shape == (2, 6), "ต่อข้างต้องได้ (2, 6) แต่ได้ %s" % (side.shape,)
    assert down.shape == (4, 3), "ต่อล่างต้องได้ (4, 3) แต่ได้ %s" % (down.shape,)
    assert np.allclose(side, [[1., 2., 3., 7., 8., 9.],
                              [4., 5., 6., 10., 11., 12.]])
    assert np.allclose(down, [[1., 2., 3.], [4., 5., 6.],
                              [7., 8., 9.], [10., 11., 12.]])


def test_concat_order_matters():
    A = np.zeros((2, 3))
    B = np.ones((2, 3))
    side, down = concat_both(A, B)
    assert np.allclose(side[:, :3], 0.) and np.allclose(side[:, 3:], 1.), \
        "A ต้องมาก่อน B"
    assert np.allclose(down[:2], 0.) and np.allclose(down[2:], 1.)


grade("2.7 concat_both", 4, test_concat_both, test_concat_order_matters)

[PASS] 2.7 concat_both                    4 / 4


---
# 3. รวมค่าด้วย axis (12 คะแนน)

`axis` คือแกนที่จะถูก **ยุบหายไป** ไม่ใช่แกนที่เก็บไว้ คนสับสนข้อนี้กันเกือบทุกคน

$$A = \begin{bmatrix}1 & 2 & 3\\ 4 & 5 & 6\end{bmatrix}_{(2,3)}$$

| เขียน | ยุบแกนไหน | ได้ | รูปร่าง |
| --- | --- | --- | --- |
| `A.sum()` | ทั้งหมด | $1{+}2{+}3{+}4{+}5{+}6 = 21$ | ตัวเลขเดียว |
| `A.sum(axis=0)` | แกน 0 คือแถว | $[1{+}4,\; 2{+}5,\; 3{+}6] = [5, 7, 9]$ | `(3,)` |
| `A.sum(axis=1)` | แกน 1 คือคอลัมน์ | $[1{+}2{+}3,\; 4{+}5{+}6] = [6, 15]$ | `(2,)` |

**วิธีจำที่ใช้ได้จริง** `axis=0` ยุบแถวทิ้ง เหลือหนึ่งค่าต่อ**คอลัมน์** ดังนั้น
`axis=0` คือ "สรุปลงมาตามแนวตั้ง" ส่วน `axis=1` คือ "สรุปไปทางขวาตามแนวนอน"

ดูจากรูปร่างก็ได้ ข้อมูล `(2, 3)` ถ้า `axis=0` ตัวเลข 2 หายไป เหลือ `(3,)`
ถ้า `axis=1` ตัวเลข 3 หายไป เหลือ `(2,)`

In [31]:
# ทายก่อนรัน: สามบรรทัดนี้ได้อะไร
guess_sum0 = "[5, 7, 9]"      # เช่น guess_sum0 = "[5, 7, 9]"
guess_sum1 = "[6, 15]"

A = np.array([[1., 2., 3.],
              [4., 5., 6.]])

print("A.sum()        =", A.sum())
print("คุณทาย axis=0 ได้", guess_sum0, " คำตอบจริง", A.sum(axis=0), " shape", A.sum(axis=0).shape)
print("คุณทาย axis=1 ได้", guess_sum1, " คำตอบจริง", A.sum(axis=1), " shape", A.sum(axis=1).shape)

A.sum()        = 21.0
คุณทาย axis=0 ได้ [5, 7, 9]  คำตอบจริง [5. 7. 9.]  shape (3,)
คุณทาย axis=1 ได้ [6, 15]  คำตอบจริง [ 6. 15.]  shape (2,)


### 3.1 `col_mean_row_mean` (4 คะแนน)

คืนค่าเฉลี่ยของแต่ละคอลัมน์ และค่าเฉลี่ยของแต่ละแถว

**ตัวอย่างที่คำนวณให้แล้ว**

$$A = \begin{bmatrix}1 & 3\\ 5 & 7\end{bmatrix}$$

$$\text{เฉลี่ยแต่ละคอลัมน์} = \left[\tfrac{1+5}{2},\; \tfrac{3+7}{2}\right] = [3, 5] \quad (2,)$$
$$\text{เฉลี่ยแต่ละแถว} = \left[\tfrac{1+3}{2},\; \tfrac{5+7}{2}\right] = [2, 6] \quad (2,)$$

ในตัวอย่างนี้ทั้งสองแบบได้รูปร่าง `(2,)` เหมือนกัน เพราะเป็นเมทริกซ์จัตุรัส แต่ตัวเลขต่างกันสิ้นเชิง
ใน test จะใช้เมทริกซ์ไม่จัตุรัสด้วย เพื่อจับว่าสลับ axis หรือเปล่า

**คำใบ้** `A.mean(axis=0)` กับ `A.mean(axis=1)` ดูตารางข้างบนว่าอันไหนคืออันไหน

In [32]:
def col_mean_row_mean(A):
    '''
    ค่าเฉลี่ยของแต่ละคอลัมน์ และของแต่ละแถว
    ใส่เข้า : A ขนาด (n, p)
    ได้ออก  : col_mean ขนาด (p,) และ row_mean ขนาด (n,)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    col_mean, row_mean = A.mean(axis=0), A.mean(axis=1)
    #########################################
    return col_mean, row_mean

In [33]:
def test_col_mean_row_mean():
    A = np.array([[1., 3.],
                  [5., 7.]])
    col_mean, row_mean = col_mean_row_mean(A)
    assert col_mean is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert np.allclose(col_mean, [3., 5.]), "ค่าเฉลี่ยของแต่ละคอลัมน์ต้องเป็น [3, 5]"
    assert np.allclose(row_mean, [2., 6.]), "ค่าเฉลี่ยของแต่ละแถวต้องเป็น [2, 6]"


def test_col_mean_row_mean_rectangular():
    A = np.array([[1., 2., 3.],
                  [3., 4., 5.]])
    col_mean, row_mean = col_mean_row_mean(A)
    assert col_mean.shape == (3,), \
        "ข้อมูล (2,3) ค่าเฉลี่ยรายคอลัมน์ต้องยาว 3 แต่ได้ %s  (สลับ axis หรือเปล่า)" % (col_mean.shape,)
    assert row_mean.shape == (2,), "ค่าเฉลี่ยรายแถวต้องยาว 2 แต่ได้ %s" % (row_mean.shape,)
    assert np.allclose(col_mean, [2., 3., 4.])
    assert np.allclose(row_mean, [2., 4.])


grade("3.1 col_mean_row_mean", 4, test_col_mean_row_mean, test_col_mean_row_mean_rectangular)

[PASS] 3.1 col_mean_row_mean              4 / 4


### 3.2 `euclidean` (4 คะแนน)

ระยะทางแบบยุคลิดระหว่างจุดสองจุด

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_k (a_k - b_k)^2}$$

**ตัวอย่างที่คำนวณให้แล้ว** `a = [0., 3.]` และ `b = [4., 0.]`

```
ผลต่าง        = [0-4, 3-0]  = [-4,  3]
ยกกำลังสอง     = [16, 9]
รวม           = 25
ถอดราก        = 5.0
```

สามเหลี่ยม 3-4-5 ที่คุ้นเคย

**คำใบ้** ต่อกันสามอย่างจากข้อ 2.1 และตารางข้างบน `np.sqrt(((a - b) ** 2).sum())`
หรือจะใช้ `np.linalg.norm(a - b)` ก็ได้ผลเดียวกัน

In [34]:
def euclidean(a, b):
    '''
    ระยะทางแบบยุคลิดระหว่างจุดสองจุด
    ใส่เข้า : a และ b ขนาด (p,) เท่ากัน
    ได้ออก  : d เป็นตัวเลขตัวเดียว
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    d = np.sqrt(((a - b) ** 2).sum())
    #########################################
    return d

In [35]:
def test_euclidean():
    d = euclidean(np.array([0., 3.]), np.array([4., 0.]))
    assert d is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert np.ndim(d) == 0, "ต้องได้ตัวเลขตัวเดียว ไม่ใช่ array (ลืม .sum() หรือเปล่า)"
    assert np.allclose(d, 5.)
    assert np.allclose(euclidean(np.zeros(3), np.zeros(3)), 0.)
    assert np.allclose(euclidean(np.array([1., 1.]), np.array([2., 2.])), np.sqrt(2.))


def test_euclidean_symmetric():
    a = np.array([1., 5., -2.])
    b = np.array([4., 1., 2.])
    assert np.allclose(euclidean(a, b), euclidean(b, a)), \
        "ระยะทางต้องเท่ากันไม่ว่าจะวัดจากฝั่งไหน (ลืมยกกำลังสองหรือเปล่า)"
    assert np.allclose(euclidean(a, b), np.sqrt(9. + 16. + 16.))


grade("3.2 euclidean", 4, test_euclidean, test_euclidean_symmetric)

[PASS] 3.2 euclidean                      4 / 4


### 3.3 `entropy_from_probs` (4 คะแนน)

$$H = -\sum_k p_k \log_2 p_k$$

**ตัวอย่างที่คำนวณให้แล้ว** `p = [0.5, 0.5]` เหรียญยุติธรรม

```
0.5 * log2(0.5) = 0.5 * (-1) = -0.5
0.5 * log2(0.5) = 0.5 * (-1) = -0.5
รวม             = -1.0
ใส่เครื่องหมายลบ  = 1.0 bit
```

**อีกตัวอย่าง** `p = [2/3, 1/3]`

$$H = -\tfrac{2}{3}\log_2 \tfrac{2}{3} - \tfrac{1}{3}\log_2 \tfrac{1}{3}
= 0.667 \times 0.585 + 0.333 \times 1.585 = 0.390 + 0.528 = 0.918$$

**และ** `p = [1.0]` ไม่มีความไม่แน่นอนเลย $H = -1 \times \log_2 1 = 0$

**คำใบ้** `-(p * np.log2(p)).sum()` ระวังวงเล็บ เครื่องหมายลบอยู่นอกสุด

In [36]:
def entropy_from_probs(p):
    '''
    เอนโทรปีจากความน่าจะเป็นที่ให้มาแล้ว
    ใส่เข้า : p ขนาด (k,) ทุกตัวมากกว่า 0 และรวมกันได้ 1
    ได้ออก  : e เป็นตัวเลขตัวเดียว หน่วยเป็นบิต
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    e = -(p * np.log2(p)).sum()
    #########################################
    return e

In [37]:
def test_entropy_from_probs():
    e = entropy_from_probs(np.array([0.5, 0.5]))
    assert e is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert np.ndim(e) == 0, "ต้องได้ตัวเลขตัวเดียว (ลืม .sum() หรือเปล่า)"
    assert np.allclose(e, 1.0), "เหรียญยุติธรรมต้องได้ 1 บิต"
    assert np.allclose(entropy_from_probs(np.array([1.0])), 0.0), \
        "แน่นอน 100 เปอร์เซ็นต์ ต้องได้ 0"
    assert np.allclose(entropy_from_probs(np.array([2. / 3, 1. / 3])), 0.918, atol=1e-3)
    assert np.allclose(entropy_from_probs(np.array([.25, .25, .25, .25])), 2.0), \
        "สี่ทางเลือกเท่า ๆ กัน ต้องได้ 2 บิต"


def test_entropy_is_positive():
    rng = np.random.default_rng(0)
    for _ in range(10):
        p = rng.random(5)
        p = p / p.sum()
        assert entropy_from_probs(p) > 0, "เอนโทรปีติดลบไม่ได้ ลืมเครื่องหมายลบหน้าวงเล็บหรือเปล่า"


grade("3.3 entropy_from_probs", 4, test_entropy_from_probs, test_entropy_is_positive)

[PASS] 3.3 entropy_from_probs             4 / 4


### 3.4 `row_max_and_argmax` (4 คะแนน)

หาค่ามากสุดของแต่ละแถว และตำแหน่งของมัน

**ตัวอย่างที่คำนวณให้แล้ว**

$$A = \begin{bmatrix}3 & 9 & 1\\ 8 & 2 & 5\end{bmatrix}$$

```
แถวที่ 0 คือ [3, 9, 1]  ->  ค่ามากสุด 9  อยู่ตำแหน่ง 1
แถวที่ 1 คือ [8, 2, 5]  ->  ค่ามากสุด 8  อยู่ตำแหน่ง 0

A.max(axis=1)     = [9, 8]
A.argmax(axis=1)  = [1, 0]
```

กฎ `axis` เหมือนข้อ 3.1 เป๊ะ คือ **แกนที่ยุบหายไป** ข้อมูล `(2,3)` ใส่ `axis=1`
ตัวเลข 3 หายไป เหลือ `(2,)` หนึ่งค่าต่อหนึ่งแถว

**คำใบ้** `A.max(axis=1)` และ `A.argmax(axis=1)` คืนทั้งคู่

In [38]:
def row_max_and_argmax(A):
    '''
    ค่ามากสุดของแต่ละแถว และตำแหน่งของมัน
    ใส่เข้า : A ขนาด (n, p)
    ได้ออก  : vals ขนาด (n,) และ idx ขนาด (n,)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    vals, idx = A.max(axis=1), A.argmax(axis=1)
    #########################################
    return vals, idx

In [39]:
def test_row_max_and_argmax():
    A = np.array([[3., 9., 1.],
                  [8., 2., 5.]])
    vals, idx = row_max_and_argmax(A)
    assert vals is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert vals.shape == (2,), "ต้องได้ค่าละหนึ่งตัวต่อแถว คือ (2,) แต่ได้ %s" % (vals.shape,)
    assert np.allclose(vals, [9., 8.])
    assert list(idx) == [1, 0], "ต้องคืนตำแหน่ง ไม่ใช่ค่า"


def test_row_max_not_column_max():
    A = np.array([[1., 2., 3.],
                  [9., 8., 7.]])
    vals, idx = row_max_and_argmax(A)
    assert np.allclose(vals, [3., 9.]), "ถ้าได้ [9,8,7] แปลว่าใช้ axis=0 ซึ่งเป็นค่ามากสุดของแต่ละคอลัมน์"


grade("3.4 row_max_and_argmax", 4, test_row_max_and_argmax, test_row_max_not_column_max)

[PASS] 3.4 row_max_and_argmax             4 / 4


---
# 4. boolean mask (16 คะแนน)

เปรียบเทียบ array กับอะไรสักอย่าง แล้วจะได้ **array ของ True กับ False** ขนาดเท่าเดิม
เรียกว่า mask แล้วเอา mask นั้นไปเลือกข้อมูลต่อได้

**ตัวอย่างที่คำนวณให้แล้ว**

```
a      = [10., 20., 30., 40.]
a > 25 = [False, False, True, True]        <- mask ขนาดเท่าเดิม
a[a > 25]      = [30., 40.]                <- เลือกเฉพาะตัวที่ True
(a > 25).sum() = 2                         <- True นับเป็น 1, False นับเป็น 0
(a > 25).mean() = 0.5                      <- สัดส่วนที่เป็นจริง
~(a > 25)      = [True, True, False, False]  <- กลับด้าน ใช้ ~ ไม่ใช่ not
```

**สามเรื่องที่ต้องจำ**

1. `True` คือ 1 และ `False` คือ 0 ดังนั้น `.sum()` คือการนับ และ `.mean()` คือสัดส่วน
2. การกลับด้านใช้ `~mask` ไม่ใช่ `not mask` คำว่า `not` ใช้กับ array ไม่ได้ จะ error
3. ถ้าจะต่อหลายเงื่อนไข ใช้ `&` และ `|` พร้อมวงเล็บครบ เช่น `(a > 10) & (a < 40)`
   ห้ามใช้ `and` กับ `or`

In [40]:
a = np.array([10., 20., 30., 40.])
mask = a > 25

print("a            =", a)
print("mask         =", mask, " shape", mask.shape)
print("a[mask]      =", a[mask])
print("mask.sum()   =", mask.sum(), " <- นับจำนวนที่จริง")
print("mask.mean()  =", mask.mean(), " <- สัดส่วนที่จริง")
print("~mask        =", ~mask)
print("a[~mask]     =", a[~mask])

a            = [10. 20. 30. 40.]
mask         = [False False  True  True]  shape (4,)
a[mask]      = [30. 40.]
mask.sum()   = 2  <- นับจำนวนที่จริง
mask.mean()  = 0.5  <- สัดส่วนที่จริง
~mask        = [ True  True False False]
a[~mask]     = [10. 20.]


### 4.1 `count_equal` (4 คะแนน)

นับว่ามีสมาชิกเท่ากับ `v` กี่ตัว

**ตัวอย่างที่คำนวณให้แล้ว**

```
Y = ['good', 'bad', 'good', 'good']
v = 'good'
Y == v  ->  [True, False, True, True]
.sum()  ->  3
```

ใช้กับตัวเลขก็ได้เหมือนกัน

```
Y = [1., 2., 1., 3.]   กับ v = 1.   ->  [True, False, True, False]  ->  2
```

**คำใบ้** `(Y == v).sum()` ครอบวงเล็บให้ครบ แล้วแปลงเป็น `int` ด้วยก็ดี

In [41]:
def count_equal(Y, v):
    '''
    นับจำนวนสมาชิกใน Y ที่เท่ากับ v
    ใส่เข้า : Y ขนาด (n,) จะเป็นตัวเลขหรือข้อความก็ได้, v คือค่าที่ต้องการนับ
    ได้ออก  : c เป็นจำนวนเต็ม
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    c = int((Y == v).sum())
    #########################################
    return c

In [42]:
def test_count_equal_strings():
    Y = np.array(['good', 'bad', 'good', 'good'])
    c = count_equal(Y, 'good')
    assert c is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert c == 3, "ต้องได้ 3 แต่ได้ %s" % c
    assert count_equal(Y, 'bad') == 1
    assert count_equal(Y, 'okay') == 0, "ถ้าไม่มีเลยต้องได้ 0 ไม่ใช่ error"


def test_count_equal_numbers():
    Y = np.array([1., 2., 1., 3.])
    assert count_equal(Y, 1.) == 2
    assert count_equal(Y, 3.) == 1


grade("4.1 count_equal", 4, test_count_equal_strings, test_count_equal_numbers)

[PASS] 4.1 count_equal                    4 / 4


### 4.2 `labels_where` (4 คะแนน)

เลือกเฉพาะ label ของข้อมูลที่ค่า attribute น้อยกว่าเกณฑ์

**ตัวอย่างที่คำนวณให้แล้ว**

```
x  = [1., 5., 2., 8.]
Y  = ['a', 'b', 'c', 'd']
th = 3.

x < 3.    ->  [True, False, True, False]
Y[x < 3.] ->  ['a', 'c']
```

สังเกตว่า mask สร้างจาก `x` แต่เอาไปใช้เลือกใน `Y` ได้ เพราะทั้งสองยาวเท่ากัน
และตำแหน่งที่ i ของทั้งคู่หมายถึงข้อมูลตัวเดียวกัน **นี่คือหัวใจของการ split ต้นไม้**

**คำใบ้** `Y[x < th]`

In [43]:
def labels_where(x, Y, th):
    '''
    เลือก label ของข้อมูลที่ค่า attribute น้อยกว่า th
    ใส่เข้า : x ขนาด (n,) เป็นตัวเลข, Y ขนาด (n,), th เป็นตัวเลข
    ได้ออก  : out คือสมาชิกของ Y เฉพาะตำแหน่งที่ x น้อยกว่า th
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = Y[x < th]
    #########################################
    return out

In [44]:
def test_labels_where():
    x = np.array([1., 5., 2., 8.])
    Y = np.array(['a', 'b', 'c', 'd'])
    out = labels_where(x, Y, 3.)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert len(out) == 2, "ต้องได้ 2 ตัว แต่ได้ %d" % len(out)
    assert list(out) == ['a', 'c'], "ต้องได้ ['a', 'c'] แต่ได้ %s" % list(out)


def test_labels_where_edges():
    x = np.array([1., 5., 2., 8.])
    Y = np.array(['a', 'b', 'c', 'd'])
    assert list(labels_where(x, Y, 0.)) == [], "ไม่มีตัวไหนผ่านเกณฑ์ ต้องได้ array ว่าง"
    assert list(labels_where(x, Y, 100.)) == ['a', 'b', 'c', 'd']
    assert list(labels_where(x, Y, 5.)) == ['a', 'c'], \
        "ใช้ < ไม่ใช่ <= ดังนั้น x เท่ากับ 5 พอดี ต้องไม่ถูกเลือก"


grade("4.2 labels_where", 4, test_labels_where, test_labels_where_edges)

[PASS] 4.2 labels_where                   4 / 4


### 4.3 `split_columns` (4 คะแนน)

แบ่งคอลัมน์ของ `X` ออกเป็นสองกอง ตาม mask

**ตัวอย่างที่คำนวณให้แล้ว** ข้อมูลเก็บแบบ `(p, n)` คือ **แถวคือ attribute คอลัมน์คือข้อมูลหนึ่งตัว**

$$X = \begin{bmatrix}1 & 2 & 3 & 4\\ 10 & 20 & 30 & 40\end{bmatrix}_{(2,4)}
\qquad \text{mask} = [T, F, T, F]$$

$$X[:, \text{mask}] = \begin{bmatrix}1 & 3\\ 10 & 30\end{bmatrix}
\qquad
X[:, \sim\text{mask}] = \begin{bmatrix}2 & 4\\ 20 & 40\end{bmatrix}$$

ต้องมี `:` กับจุลภาคข้างหน้า ถ้าเขียน `X[mask]` เฉย ๆ มันจะไปเลือก**แถว**
ซึ่งผิด เพราะแถวคือ attribute ไม่ใช่ข้อมูล

**คำใบ้** `X[:, mask]` และ `X[:, ~mask]`

In [45]:
def split_columns(X, mask):
    '''
    แบ่งคอลัมน์ของ X ออกเป็นสองกองตาม mask
    ใส่เข้า : X ขนาด (p, n), mask เป็น boolean ขนาด (n,)
    ได้ออก  : left คือคอลัมน์ที่ mask เป็น True, right คือที่เหลือ
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    left, right = X[:, mask], X[:, ~mask]
    #########################################
    return left, right

In [46]:
def test_split_columns():
    X = np.array([[1., 2., 3., 4.],
                  [10., 20., 30., 40.]])
    mask = np.array([True, False, True, False])
    left, right = split_columns(X, mask)
    assert left is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert left.shape == (2, 2), \
        "ต้องได้ (2, 2) แต่ได้ %s  (ลืม : กับจุลภาค แล้วไปเลือกแถวหรือเปล่า)" % (left.shape,)
    assert np.allclose(left, [[1., 3.], [10., 30.]])
    assert np.allclose(right, [[2., 4.], [20., 40.]])


def test_split_columns_all_one_side():
    X = np.arange(6.).reshape(2, 3)
    left, right = split_columns(X, np.array([True, True, True]))
    assert left.shape == (2, 3) and right.shape == (2, 0), \
        "ถ้า mask เป็นจริงหมด กองขวาต้องว่าง แต่ยังต้องมี 2 แถวอยู่"


grade("4.3 split_columns", 4, test_split_columns, test_split_columns_all_one_side)

[PASS] 4.3 split_columns                  4 / 4


### 4.4 `accuracy` (4 คะแนน)

สัดส่วนของการทำนายที่ถูกต้อง

**ตัวอย่างที่คำนวณให้แล้ว**

```
y     = ['a', 'b', 'a', 'c', 'a']
yhat  = ['a', 'b', 'c', 'c', 'a']
y == yhat  ->  [True, True, False, True, True]
.sum()     ->  4
.mean()    ->  4/5 = 0.8
```

**คำใบ้** `(y == yhat).mean()` หรือจะเขียน `.sum() / len(y)` ก็ได้ผลเดียวกัน

In [47]:
def accuracy(y, yhat):
    '''
    สัดส่วนของการทำนายที่ถูกต้อง
    ใส่เข้า : y และ yhat ขนาด (n,) เท่ากัน
    ได้ออก  : acc เป็นตัวเลขระหว่าง 0 ถึง 1
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    acc = (y == yhat).mean()
    #########################################
    return acc

In [48]:
def test_accuracy():
    y = np.array(['a', 'b', 'a', 'c', 'a'])
    yhat = np.array(['a', 'b', 'c', 'c', 'a'])
    acc = accuracy(y, yhat)
    assert acc is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert np.allclose(acc, 0.8), "ต้องได้ 0.8 แต่ได้ %s  (ลืมหารด้วยจำนวนหรือเปล่า)" % acc


def test_accuracy_edges():
    y = np.array([1., 2., 3.])
    assert np.allclose(accuracy(y, y), 1.0), "ทายถูกหมดต้องได้ 1.0"
    assert np.allclose(accuracy(y, np.array([9., 9., 9.])), 0.0), "ทายผิดหมดต้องได้ 0.0"
    assert np.allclose(accuracy(y, np.array([1., 9., 9.])), 1. / 3)


grade("4.4 accuracy", 4, test_accuracy, test_accuracy_edges)

[PASS] 4.4 accuracy                       4 / 4


### 4.5 `one_hot` (4 คะแนน)

แปลง label ที่เป็นตัวเลข ให้เป็นเวกเตอร์ที่มีเลข 1 อยู่ตำแหน่งเดียว

**ตัวอย่างที่คำนวณให้แล้ว** มี 4 คลาส

$$\texttt{np.eye(4)} = \begin{bmatrix}
1&0&0&0\\ 0&1&0&0\\ 0&0&1&0\\ 0&0&0&1\end{bmatrix}$$

```
label 2  ->  np.eye(4)[2]  =  [0, 0, 1, 0]
label 0  ->  np.eye(4)[0]  =  [1, 0, 0, 0]
```

เคล็ดลับคือ **แถวที่ i ของเมทริกซ์เอกลักษณ์ ก็คือ one-hot ของคลาส i พอดี**
และถ้าส่ง array ของ label เข้าไปทีเดียว จะได้ทั้งตารางเลย

```
Y = [2, 0, 1]  ->  np.eye(4)[Y]  =  [[0, 0, 1, 0],
                                     [1, 0, 0, 0],
                                     [0, 1, 0, 0]]     ขนาด (3, 4)
```

**คำใบ้** `np.eye(c)[y]` บรรทัดเดียว ใช้ได้ทั้งกับ label ตัวเดียวและทั้ง array

In [49]:
def one_hot(y, c):
    '''
    แปลง label เป็น one-hot
    ใส่เข้า : y เป็นจำนวนเต็มตัวเดียว หรือ array ของจำนวนเต็มความยาว n, c คือจำนวนคลาส
    ได้ออก  : out ขนาด (c,) ถ้า y เป็นตัวเดียว หรือ (n, c) ถ้า y เป็น array
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    out = np.eye(c)[y]
    #########################################
    return out

In [50]:
def test_one_hot_single():
    out = one_hot(2, 4)
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert out.shape == (4,)
    assert np.allclose(out, [0., 0., 1., 0.])
    assert np.allclose(one_hot(0, 3), [1., 0., 0.])


def test_one_hot_batch():
    out = one_hot(np.array([2, 0, 1]), 4)
    assert out.shape == (3, 4), "ส่ง array เข้าไปต้องได้ตาราง (n, c)"
    assert np.allclose(out, [[0, 0, 1, 0], [1, 0, 0, 0], [0, 1, 0, 0]])
    assert np.allclose(out.sum(axis=1), 1.), "แต่ละแถวต้องมีเลข 1 ตัวเดียว"


grade("4.5 one_hot", 4, test_one_hot_single, test_one_hot_batch)

[PASS] 4.5 one_hot                        4 / 4


---
# 5. unique, argmax, argsort (16 คะแนน)

กลุ่มนี้ตอบคำถามว่า "มีค่าอะไรบ้าง", "อันไหนมากสุด", "เรียงแล้วใครมาก่อน"

**สิ่งที่ต้องแยกให้ออก คำว่า `arg` แปลว่า ตำแหน่ง ไม่ใช่ค่า**

```
a = [30., 10., 20.]

a.max()      = 30.0      <- ค่าที่มากที่สุด
np.argmax(a) = 0         <- ตำแหน่งของค่าที่มากที่สุด

np.sort(a)    = [10., 20., 30.]   <- ค่าที่เรียงแล้ว
np.argsort(a) = [1, 2, 0]         <- ตำแหน่งเดิม เรียงตามค่าจากน้อยไปมาก
```

อ่าน `np.argsort(a) = [1, 2, 0]` ว่า "ตัวที่เล็กที่สุดอยู่ตำแหน่ง 1, ถัดมาอยู่ตำแหน่ง 2,
ใหญ่สุดอยู่ตำแหน่ง 0"

ที่ต้องใช้ `arg` เพราะส่วนใหญ่เราไม่ได้อยากได้ระยะทางที่สั้นที่สุด เราอยากรู้ว่า
**ข้อมูลตัวไหน**ที่ใกล้ที่สุด แล้วเอาตำแหน่งนั้นไปดู label ต่อ

In [51]:
a = np.array([30., 10., 20.])
print("a             =", a)
print("a.max()       =", a.max(), "   <- ค่า")
print("np.argmax(a)  =", np.argmax(a), "    <- ตำแหน่ง")
print("np.sort(a)    =", np.sort(a))
print("np.argsort(a) =", np.argsort(a))
print()

Y = np.array(['good', 'bad', 'good', 'ok', 'good'])
vals, counts = np.unique(Y, return_counts=True)
print("Y      =", Y)
print("vals   =", vals, "   <- เรียงตามตัวอักษรเสมอ")
print("counts =", counts)

a             = [30. 10. 20.]
a.max()       = 30.0    <- ค่า
np.argmax(a)  = 0     <- ตำแหน่ง
np.sort(a)    = [10. 20. 30.]
np.argsort(a) = [1 2 0]

Y      = ['good' 'bad' 'good' 'ok' 'good']
vals   = ['bad' 'good' 'ok']    <- เรียงตามตัวอักษรเสมอ
counts = [1 3 1]


### 5.1 `distinct_values` (4 คะแนน)

ค่าที่ไม่ซ้ำกัน เรียงจากน้อยไปมาก

**ตัวอย่างที่คำนวณให้แล้ว**

```
x = [3., 1., 3., 2., 1.]
np.unique(x) = [1., 2., 3.]        <- ตัดตัวซ้ำออก แล้วเรียงให้ด้วย
```

ใช้กับข้อความได้เหมือนกัน เรียงตามตัวอักษร

```
Y = ['good', 'bad', 'good']
np.unique(Y) = ['bad', 'good']
```

**คำใบ้** `np.unique(x)`

In [52]:
def distinct_values(x):
    '''
    ค่าที่ไม่ซ้ำกัน เรียงจากน้อยไปมาก
    ใส่เข้า : x ขนาด (n,)
    ได้ออก  : out ขนาด (k,) เมื่อ k คือจำนวนค่าที่ต่างกัน
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    out = np.unique(x)
    #########################################
    return out

In [53]:
def test_distinct_values():
    out = distinct_values(np.array([3., 1., 3., 2., 1.]))
    assert out is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert np.allclose(out, [1., 2., 3.]), "ต้องตัดตัวซ้ำออกและเรียงจากน้อยไปมาก"
    assert len(distinct_values(np.array([5., 5., 5.]))) == 1


def test_distinct_values_strings():
    out = distinct_values(np.array(['good', 'bad', 'good']))
    assert list(out) == ['bad', 'good'], "ข้อความเรียงตามตัวอักษร"


grade("5.1 distinct_values", 4, test_distinct_values, test_distinct_values_strings)

[PASS] 5.1 distinct_values                4 / 4


### 5.2 `value_counts` (4 คะแนน)

ค่าที่ไม่ซ้ำ พร้อมจำนวนของแต่ละค่า

**ตัวอย่างที่คำนวณให้แล้ว**

```
Y = ['good', 'bad', 'good', 'ok', 'good']

vals   = ['bad', 'good', 'ok']
counts = [1,     3,      1   ]
```

อ่านคู่กันตามตำแหน่ง `vals[1]` คือ `'good'` และ `counts[1]` คือ 3 แปลว่า good มี 3 ตัว

**คำใบ้** `np.unique(Y, return_counts=True)` คืนมาเป็นคู่อยู่แล้ว

In [54]:
def value_counts(Y):
    '''
    ค่าที่ไม่ซ้ำ พร้อมจำนวนของแต่ละค่า
    ใส่เข้า : Y ขนาด (n,)
    ได้ออก  : vals ขนาด (k,) และ counts ขนาด (k,) อ่านคู่กันตามตำแหน่ง
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    vals, counts = np.unique(Y, return_counts=True)
    #########################################
    return vals, counts

In [55]:
def test_value_counts():
    Y = np.array(['good', 'bad', 'good', 'ok', 'good'])
    vals, counts = value_counts(Y)
    assert vals is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert list(vals) == ['bad', 'good', 'ok']
    assert list(counts) == [1, 3, 1], "ต้องได้ [1, 3, 1] แต่ได้ %s" % list(counts)
    assert counts.sum() == len(Y), "จำนวนรวมต้องเท่ากับความยาวของ Y"


def test_value_counts_to_probabilities():
    Y = np.array([1., 1., 2., 2., 2., 2.])
    vals, counts = value_counts(Y)
    p = counts / len(Y)
    assert np.allclose(p, [1. / 3, 2. / 3]), \
        "เอา counts หารด้วยจำนวนทั้งหมด จะได้ความน่าจะเป็นพร้อมใส่สูตรเอนโทรปี"


grade("5.2 value_counts", 4, test_value_counts, test_value_counts_to_probabilities)

[PASS] 5.2 value_counts                   4 / 4


### 5.3 `most_frequent` (4 คะแนน)

label ที่พบบ่อยที่สุด

**ตัวอย่างที่คำนวณให้แล้ว**

```
Y = ['good', 'bad', 'good', 'ok', 'good']

ขั้นที่ 1  vals, counts = ['bad', 'good', 'ok'], [1, 3, 1]
ขั้นที่ 2  np.argmax(counts) = 1          <- ตำแหน่งของจำนวนที่มากสุด
ขั้นที่ 3  vals[1] = 'good'               <- เอาตำแหน่งไปหยิบค่า
```

สามขั้นนี้คือลวดลายที่จะเจอซ้ำทั้งเทอม คือ **นับ แล้วหาตำแหน่งของตัวมากสุด แล้วเอาตำแหน่งไปหยิบค่า**

**คำใบ้** ใช้ข้อ 5.2 แล้วต่อด้วย `np.argmax`

In [56]:
def most_frequent(Y):
    '''
    label ที่พบบ่อยที่สุด ถ้าเท่ากันจะเอาตัวไหนก็ได้
    ใส่เข้า : Y ขนาด (n,)
    ได้ออก  : y คือค่าหนึ่งค่า ไม่ใช่ array
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    vals, counts = value_counts(Y)
    y = vals[np.argmax(counts)]
    #########################################
    return y

In [57]:
def test_most_frequent():
    y = most_frequent(np.array(['good', 'bad', 'good', 'ok', 'good']))
    assert y is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert y == 'good', "ต้องได้ 'good' แต่ได้ %r" % (y,)
    assert np.ndim(y) == 0, "ต้องได้ค่าเดียว ไม่ใช่ array (ลืมเอา argmax ไปหยิบจาก vals หรือเปล่า)"


def test_most_frequent_numbers():
    assert most_frequent(np.array([1., 2., 2., 2., 3.])) == 2.
    assert most_frequent(np.array([7.])) == 7.
    assert most_frequent(np.array(['a', 'b', 'b'])) == 'b'


grade("5.3 most_frequent", 4, test_most_frequent, test_most_frequent_numbers)

[PASS] 5.3 most_frequent                  4 / 4


### 5.4 `k_nearest_index` (4 คะแนน)

ตำแหน่งของ `k` ตัวที่มีค่าน้อยที่สุด

**ตัวอย่างที่คำนวณให้แล้ว** `d` คือระยะทางจากจุดที่เราสนใจไปยังข้อมูลฝึกทั้ง 5 ตัว

```
d = [0.9, 0.2, 0.5, 0.1, 0.7]
      0    1    2    3    4      <- ตำแหน่ง

np.argsort(d)      = [3, 1, 2, 4, 0]      ใกล้สุดคือตำแหน่ง 3 (ค่า 0.1)
np.argsort(d)[:3]  = [3, 1, 2]            เอาสามตัวแรก
```

**คำใบ้** `np.argsort(d)[:k]`

In [58]:
def k_nearest_index(d, k):
    '''
    ตำแหน่งของ k ตัวที่มีค่าน้อยที่สุด เรียงจากน้อยไปมาก
    ใส่เข้า : d ขนาด (n,), k เป็นจำนวนเต็ม
    ได้ออก  : idx ขนาด (k,) เป็นตำแหน่ง ไม่ใช่ค่า
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    idx = np.argsort(d)[:k]
    #########################################
    return idx

In [59]:
def test_k_nearest_index():
    d = np.array([0.9, 0.2, 0.5, 0.1, 0.7])
    idx = k_nearest_index(d, 3)
    assert idx is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert len(idx) == 3, "ขอ k=3 ต้องได้ 3 ตำแหน่ง แต่ได้ %d" % len(idx)
    assert list(idx) == [3, 1, 2], \
        "ต้องได้ [3, 1, 2] แต่ได้ %s  (คืนตำแหน่ง ไม่ใช่ค่า และเรียงจากใกล้ไปไกล)" % list(idx)
    assert list(k_nearest_index(d, 1)) == [3]


def test_k_nearest_index_then_vote():
    '''นี่คือสิ่งที่ Lab 2 จะทำต่อจากข้อนี้'''
    d = np.array([0.9, 0.2, 0.5, 0.1, 0.7])
    Ytrain = np.array(['a', 'b', 'b', 'b', 'a'])
    idx = k_nearest_index(d, 3)
    assert list(Ytrain[idx]) == ['b', 'b', 'b'], \
        "เอาตำแหน่งไปหยิบ label ได้ด้วย Ytrain[idx]"


grade("5.4 k_nearest_index", 4, test_k_nearest_index, test_k_nearest_index_then_vote)

[PASS] 5.4 k_nearest_index                4 / 4


---
# 6. พีชคณิตเชิงเส้น (16 คะแนน)

### `@` กับ `*` ไม่เหมือนกัน

`*` คูณทีละตัว ส่วน `@` คือการคูณเมทริกซ์จริง ๆ

$$A = \begin{bmatrix}1 & 2\\ 3 & 4\end{bmatrix}
\qquad
B = \begin{bmatrix}5 & 6\\ 7 & 8\end{bmatrix}$$

$$A \ast B = \begin{bmatrix}1{\times}5 & 2{\times}6\\ 3{\times}7 & 4{\times}8\end{bmatrix}
= \begin{bmatrix}5 & 12\\ 21 & 32\end{bmatrix}
\qquad\text{คูณตรงตำแหน่ง}$$

$$A \mathbin{@} B = \begin{bmatrix}1{\times}5 + 2{\times}7 & 1{\times}6 + 2{\times}8\\
3{\times}5 + 4{\times}7 & 3{\times}6 + 4{\times}8\end{bmatrix}
= \begin{bmatrix}19 & 22\\ 43 & 50\end{bmatrix}
\qquad\text{แถวคูณคอลัมน์}$$

**กฎรูปร่างของ `@`** ตัวในต้องเท่ากัน ตัวนอกคือคำตอบ

$$(n, p) \mathbin{@} (p, 1) \;\longrightarrow\; (n, 1)$$

ถ้าตัวในไม่เท่ากัน NumPy จะ error ทันที ซึ่งเป็นเรื่องดี เพราะมันบอกเราว่าทำผิด
ต่างจาก `*` ที่อาจจะ broadcasting ไปเงียบ ๆ แล้วได้คำตอบผิดโดยไม่ฟ้อง

In [60]:
# ทายก่อนรัน: A * B กับ A @ B ได้ต่างกันอย่างไร
guess_star = "[[5, 12], [21, 32]]"     # เช่น guess_star = "[[5, 12], [21, 32]]"
guess_at = "[[19, 22], [43, 50]]"

A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])

print("คุณทาย A * B =", guess_star)
print("คำตอบจริง"); print(A * B)
print()
print("คุณทาย A @ B =", guess_at)
print("คำตอบจริง"); print(A @ B)
print()
X = np.ones((5, 3))
w = np.ones((3, 1))
print("(5,3) @ (3,1) ->", (X @ w).shape, "  ตัวใน 3 ตรงกัน ตัวนอกคือ 5 กับ 1")

คุณทาย A * B = [[5, 12], [21, 32]]
คำตอบจริง
[[ 5. 12.]
 [21. 32.]]

คุณทาย A @ B = [[19, 22], [43, 50]]
คำตอบจริง
[[19. 22.]
 [43. 50.]]

(5,3) @ (3,1) -> (5, 1)   ตัวใน 3 ตรงกัน ตัวนอกคือ 5 กับ 1


### 6.1 `linear_model` (4 คะแนน)

คำทำนายของโมเดลเชิงเส้น

$$\hat{\mathbf{y}} = X\mathbf{w} + b$$

**ตัวอย่างที่คำนวณให้แล้ว**

$$X = \begin{bmatrix}1 & 2\\ 3 & 4\end{bmatrix},\quad
\mathbf{w} = \begin{bmatrix}1\\ 1\end{bmatrix},\quad b = 0.5$$

$$X\mathbf{w} = \begin{bmatrix}1{\times}1 + 2{\times}1\\ 3{\times}1 + 4{\times}1\end{bmatrix}
= \begin{bmatrix}3\\ 7\end{bmatrix}
\qquad
X\mathbf{w} + b = \begin{bmatrix}3.5\\ 7.5\end{bmatrix}$$

`b` เป็นตัวเลขตัวเดียว แต่บวกเข้ากับ array ได้ เพราะ broadcasting ยืดให้เอง

**คำใบ้** `X @ w + b` บรรทัดเดียว

In [61]:
def linear_model(X, w, b):
    '''
    คำทำนายของโมเดลเชิงเส้น
    ใส่เข้า : X ขนาด (n, p), w ขนาด (p, 1), b เป็นตัวเลขตัวเดียว
    ได้ออก  : yhat ขนาด (n, 1)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    yhat = X @ w + b
    #########################################
    return yhat

In [62]:
def test_linear_model():
    X = np.array([[1., 2.],
                  [3., 4.]])
    w = np.array([[1.], [1.]])
    yhat = linear_model(X, w, 0.5)
    assert yhat is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert yhat.shape == (2, 1), \
        "ต้องได้ (2, 1) แต่ได้ %s  (ใช้ @ หรือ * ลองดูอีกที)" % (yhat.shape,)
    assert np.allclose(yhat, [[3.5], [7.5]])


def test_linear_model_shapes():
    X = np.ones((5, 3))
    w = np.array([[1.], [2.], [3.]])
    yhat = linear_model(X, w, 0.)
    assert yhat.shape == (5, 1)
    assert np.allclose(yhat, 6.), "ทุกแถวของ X เป็น 1 หมด ดังนั้นทุกคำทำนายคือ 1+2+3"


grade("6.1 linear_model", 4, test_linear_model, test_linear_model_shapes)

[PASS] 6.1 linear_model                   4 / 4


### 6.2 `gram` (4 คะแนน)

$$G = X^\top X$$

**ตัวอย่างที่คำนวณให้แล้ว**

$$X = \begin{bmatrix}1 & 0\\ 0 & 1\\ 1 & 1\end{bmatrix}_{(3,2)}
\qquad
X^\top = \begin{bmatrix}1 & 0 & 1\\ 0 & 1 & 1\end{bmatrix}_{(2,3)}$$

$$X^\top X = \begin{bmatrix}1{+}0{+}1 & 0{+}0{+}1\\ 0{+}0{+}1 & 0{+}1{+}1\end{bmatrix}
= \begin{bmatrix}2 & 1\\ 1 & 2\end{bmatrix}_{(2,2)}$$

สังเกตรูปร่าง $(2,3) \mathbin{@} (3,2) \to (2,2)$ ตัว $n$ หายไป เหลือแต่ $p$
แปลว่าไม่ว่าข้อมูลจะมีกี่แสนแถว เมทริกซ์นี้ก็เล็กเท่าจำนวน feature เสมอ
นี่คือเหตุผลที่ normal equation แก้ได้เร็ว

**คำใบ้** `X.T @ X` อย่าสลับเป็น `X @ X.T` จะได้ `(n, n)` ซึ่งใหญ่กว่ามากและไม่ใช่สิ่งที่ต้องการ

In [63]:
def gram(X):
    '''
    เมทริกซ์ X ทรานสโพส คูณ X
    ใส่เข้า : X ขนาด (n, p)
    ได้ออก  : G ขนาด (p, p)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    G = X.T @ X
    #########################################
    return G

In [64]:
def test_gram():
    X = np.array([[1., 0.],
                  [0., 1.],
                  [1., 1.]])
    G = gram(X)
    assert G is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert G.shape == (2, 2), \
        "ต้องได้ (2, 2) แต่ได้ %s  (สลับเป็น X @ X.T หรือเปล่า)" % (G.shape,)
    assert np.allclose(G, [[2., 1.], [1., 2.]])


def test_gram_is_symmetric():
    rng = np.random.default_rng(0)
    X = rng.normal(0., 1., (10, 4))
    G = gram(X)
    assert G.shape == (4, 4)
    assert np.allclose(G, G.T), "X.T @ X ต้องสมมาตรเสมอ"


grade("6.2 gram", 4, test_gram, test_gram_is_symmetric)

[PASS] 6.2 gram                           4 / 4


### 6.3 `ridge_matrix` (4 คะแนน)

$$X^\top X + \alpha I$$

**ตัวอย่างที่คำนวณให้แล้ว** ใช้ $G$ จากข้อที่แล้ว กับ $\alpha = 10$

$$\begin{bmatrix}2 & 1\\ 1 & 2\end{bmatrix}
+ 10\begin{bmatrix}1 & 0\\ 0 & 1\end{bmatrix}
= \begin{bmatrix}12 & 1\\ 1 & 12\end{bmatrix}$$

การบวก $\alpha$ ลงบนเส้นทแยงมุมคือสิ่งที่ ridge regression ทำ และมันมีผลสองอย่างพร้อมกัน
ในทางสถิติมันคือการใส่ prior ให้ $\mathbf{w}$ ในทางตัวเลขมันทำให้เมทริกซ์ invert ได้เสมอ
แม้ feature จะสัมพันธ์กันจนเกือบ singular

**คำใบ้** `np.eye(p)` คือเมทริกซ์เอกลักษณ์ขนาด `p` และ `p` หาได้จาก `X.shape[1]`

In [65]:
def ridge_matrix(X, alpha):
    '''
    X ทรานสโพส คูณ X แล้วบวก alpha ลงบนเส้นทแยงมุม
    ใส่เข้า : X ขนาด (n, p), alpha เป็นตัวเลข
    ได้ออก  : A ขนาด (p, p)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    A = X.T @ X + alpha * np.eye(X.shape[1])
    #########################################
    return A

In [66]:
def test_ridge_matrix():
    X = np.array([[1., 0.],
                  [0., 1.],
                  [1., 1.]])
    A = ridge_matrix(X, 10.)
    assert A is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert A.shape == (2, 2)
    assert np.allclose(A, [[12., 1.], [1., 12.]]), \
        "alpha ต้องบวกเฉพาะเส้นทแยงมุม ไม่ใช่บวกทุกช่อง"


def test_ridge_matrix_alpha_zero():
    X = np.array([[1., 0.],
                  [0., 1.],
                  [1., 1.]])
    assert np.allclose(ridge_matrix(X, 0.), [[2., 1.], [1., 2.]]), \
        "alpha เป็น 0 ต้องได้ X.T @ X เฉย ๆ"


grade("6.3 ridge_matrix", 4, test_ridge_matrix, test_ridge_matrix_alpha_zero)

[PASS] 6.3 ridge_matrix                   4 / 4


### 6.4 `solve` (4 คะแนน)

แก้ระบบสมการ $A\mathbf{z} = \mathbf{b}$ หา $\mathbf{z}$

**ตัวอย่างที่คำนวณให้แล้ว**

$$\begin{bmatrix}2 & 0\\ 0 & 4\end{bmatrix}\mathbf{z} = \begin{bmatrix}6\\ 8\end{bmatrix}$$

แก้ด้วยมือได้เลย $2z_1 = 6$ ดังนั้น $z_1 = 3$ และ $4z_2 = 8$ ดังนั้น $z_2 = 2$

$$\mathbf{z} = \begin{bmatrix}3\\ 2\end{bmatrix}$$

**ทำไมไม่ใช้ `np.linalg.inv(A) @ b`** ได้คำตอบเดียวกัน แต่ `solve` แม่นยำกว่าและเร็วกว่า
เพราะมันไม่ต้องสร้างเมทริกซ์ผกผันขึ้นมาจริง ๆ ซึ่งเป็นขั้นตอนที่สะสมความคลาดเคลื่อน
ในแลปจะเขียนแบบไหนก็ได้ แต่ในงานจริงให้ใช้ `solve`

**คำใบ้** `np.linalg.solve(A, b)`

In [67]:
def solve(A, b):
    '''
    แก้ระบบสมการ A z = b
    ใส่เข้า : A ขนาด (p, p), b ขนาด (p, 1)
    ได้ออก  : z ขนาด (p, 1)
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    z = np.linalg.solve(A, b)
    #########################################
    return z

In [68]:
def test_solve():
    A = np.array([[2., 0.],
                  [0., 4.]])
    b = np.array([[6.], [8.]])
    z = solve(A, b)
    assert z is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert z.shape == (2, 1), "ต้องได้ (2, 1) แต่ได้ %s" % (z.shape,)
    assert np.allclose(z, [[3.], [2.]])


def test_solve_checks_out():
    rng = np.random.default_rng(0)
    A = rng.normal(0., 1., (4, 4)) + 4. * np.eye(4)
    b = rng.normal(0., 1., (4, 1))
    z = solve(A, b)
    assert np.allclose(A @ z, b, atol=1e-8), \
        "เอาคำตอบคูณกลับเข้าไปใน A ต้องได้ b กลับมา"


grade("6.4 solve", 4, test_solve, test_solve_checks_out)

[PASS] 6.4 solve                          4 / 4


### 6.5 `softmax_jacobian` (4 คะแนน)

อนุพันธ์ของ softmax เทียบกับ logit เป็นเมทริกซ์ ไม่ใช่เวกเตอร์ เพราะ $a_i$ ทุกตัวขึ้นกับ $z_j$ ทุกตัว

$$\frac{\partial a_i}{\partial z_j} = \begin{cases}
a_i(1 - a_i) & i = j\\
-a_i a_j & i \neq j
\end{cases}
\qquad\Longleftrightarrow\qquad
J = \operatorname{diag}(\mathbf{a}) - \mathbf{a}\mathbf{a}^\top$$

**ตัวอย่างที่คำนวณให้แล้ว** $\mathbf{a} = [0.2,\; 0.3,\; 0.5]^\top$

$$\operatorname{diag}(\mathbf{a}) = \begin{bmatrix}.2&0&0\\ 0&.3&0\\ 0&0&.5\end{bmatrix}
\qquad
\mathbf{a}\mathbf{a}^\top = \begin{bmatrix}.04&.06&.10\\ .06&.09&.15\\ .10&.15&.25\end{bmatrix}$$

$$J = \begin{bmatrix}.16&-.06&-.10\\ -.06&.21&-.15\\ -.10&-.15&.25\end{bmatrix}$$

ตรวจด้วยมือ ช่องทแยง $0.2(1-0.2) = 0.16$ ถูก ช่องนอกทแยง $-0.2 \times 0.3 = -0.06$ ถูก

**สมบัติที่ใช้ตรวจคำตอบได้** ผลรวมของแต่ละแถวต้องเป็นศูนย์
$0.16 - 0.06 - 0.10 = 0$ เพราะความน่าจะเป็นรวมกันได้ 1 เสมอ ขยับ logit แล้วผลรวมต้องไม่เปลี่ยน

**คำใบ้**
- `np.diag(v)` เมื่อ `v` เป็น 1 มิติ จะสร้างเมทริกซ์ทแยง ดังนั้นต้อง `a.reshape(-1)` ก่อน
- `a @ a.T` เมื่อ `a` เป็น `(c, 1)` จะได้ `(c, c)` ซึ่งคือ outer product

In [69]:
def softmax_jacobian(a):
    '''
    เมทริกซ์อนุพันธ์ของ softmax เทียบกับ logit
    ใส่เข้า : a ขนาด (c, 1) ค่า softmax ที่รวมกันได้ 1
    ได้ออก  : J ขนาด (c, c) โดย J[i,j] = d a[i] / d z[j]
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    J = np.diag(a.reshape(-1)) - a @ a.T
    #########################################
    return J

In [70]:
def test_softmax_jacobian():
    a = np.array([[0.2], [0.3], [0.5]])
    J = softmax_jacobian(a)
    assert J is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert J.shape == (3, 3), "ต้องได้ (3, 3) แต่ได้ %s" % (J.shape,)
    assert np.allclose(np.diag(J), [0.16, 0.21, 0.25]), "ช่องทแยงต้องเป็น a[i](1-a[i])"
    assert np.allclose(J[0, 1], -0.06) and np.allclose(J[0, 2], -0.10)
    assert np.allclose(J, J.T), "เมทริกซ์นี้ต้องสมมาตร"


def test_softmax_jacobian_rows_sum_to_zero():
    rng = np.random.default_rng(0)
    for _ in range(10):
        v = rng.random((5, 1))
        a = v / v.sum()
        assert np.allclose(softmax_jacobian(a).sum(axis=1), 0., atol=1e-12), \
            "ผลรวมของแต่ละแถวต้องเป็นศูนย์ เพราะความน่าจะเป็นรวมกันได้ 1 เสมอ"


grade("6.5 softmax_jacobian", 4, test_softmax_jacobian, test_softmax_jacobian_rows_sum_to_zero)

[PASS] 6.5 softmax_jacobian               4 / 4


---
# 7. การสุ่ม (8 คะแนน)

สองแบบที่ต้องแยกให้ออก

| | ฟังก์ชัน | ได้ค่าซ้ำไหม | ใช้ทำอะไร |
| --- | --- | --- | --- |
| **คืนที่** | `np.random.randint(n, size=k)` | ซ้ำได้ | bootstrap ใน Lab 4 |
| **ไม่คืนที่** | `np.random.choice(p, m, replace=False)` | ไม่ซ้ำ | สุ่มเลือก attribute ใน random forest |

**ตัวอย่างที่คำนวณให้แล้ว** สมมติ `n = 5`

```
np.random.randint(5, size=5)              ->  [3, 0, 3, 1, 3]   <- เลข 3 โผล่สามครั้ง ปกติ
np.random.choice(5, 2, replace=False)     ->  [4, 1]            <- ไม่มีทางซ้ำ
```

### การตรึงผลให้ซ้ำได้

ถ้าไม่ตรึง เรารันสองครั้งได้คนละคำตอบ แล้วเทียบผลไม่ได้ว่าโมเดลดีขึ้นจริงหรือแค่โชคดี
ใช้ `np.random.seed(0)` ก่อนเริ่ม แล้วลำดับการสุ่มจะเหมือนเดิมทุกครั้ง

```python
np.random.seed(0)
print(np.random.randint(10, size=3))      # ได้เลขชุดเดิมทุกครั้งที่รัน
```

ในงานวิจัยจริง ทุกการทดลองต้องรันซ้ำได้ นี่เป็นกติกาข้อหนึ่งของวิชานี้

**หมายเหตุ** NumPy รุ่นใหม่แนะนำให้ใช้ `rng = np.random.default_rng(0)` แล้วเรียก
`rng.integers(...)` แทน ซึ่งดีกว่าในงานจริง แต่ในแลปนี้เราใช้แบบ `np.random.*`
เพราะฟังก์ชันใน Lab 4 ไม่ได้รับ `rng` เข้าไปเป็นพารามิเตอร์

In [71]:
np.random.seed(0)
print("คืนที่      ", np.random.randint(5, size=5))
print("ไม่คืนที่    ", np.random.choice(5, 2, replace=False))

np.random.seed(0)
print("ตรึง seed แล้วรันใหม่")
print("คืนที่      ", np.random.randint(5, size=5), "  <- เหมือนเดิมเป๊ะ")

คืนที่       [4 0 3 3 3]
ไม่คืนที่     [2 0]
ตรึง seed แล้วรันใหม่
คืนที่       [4 0 3 3 3]   <- เหมือนเดิมเป๊ะ


### 7.1 `bootstrap_index` (4 คะแนน)

สุ่มตำแหน่งแบบคืนที่ ให้ได้จำนวนเท่าเดิมกับข้อมูลต้นฉบับ

**ตัวอย่างที่คำนวณให้แล้ว** ข้อมูลมี 5 ตัว

```
idx = [3, 0, 3, 1, 3]       <- ยาว 5 เท่าเดิม และมีตัวซ้ำได้

เอาไปใช้ต่อ
X[:, idx]   เลือกคอลัมน์ตามตำแหน่งนี้
Y[idx]      เลือก label ตามตำแหน่งเดียวกัน
```

**ต้องใช้ `idx` ชุดเดียวกันทั้ง X และ Y** ถ้าสุ่มสองครั้งแยกกัน label จะไม่ตรงกับข้อมูล
แล้วโมเดลจะเรียนจากของที่จับคู่ผิด ซึ่งเป็นบั๊กที่ test จับไม่ได้ง่าย ๆ

ทำไมต้องคืนที่ เพราะอยากให้แต่ละต้นเห็นข้อมูลชุดที่ต่างกันเล็กน้อย โดยยังมีขนาดเท่าเดิม
โดยเฉลี่ยแต่ละ bootstrap จะเห็นข้อมูลจริงประมาณ 63 เปอร์เซ็นต์ ที่เหลือคือตัวซ้ำ

**คำใบ้** `np.random.randint(n, size=n)`

In [72]:
def bootstrap_index(n):
    '''
    สุ่มตำแหน่งแบบคืนที่ จำนวน n ตำแหน่ง จากช่วง 0 ถึง n-1
    ใส่เข้า : n เป็นจำนวนเต็ม
    ได้ออก  : idx ขนาด (n,) เป็นจำนวนเต็ม มีค่าซ้ำได้
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    idx = np.random.randint(n, size=n)
    #########################################
    return idx

In [73]:
def test_bootstrap_index():
    np.random.seed(0)
    idx = bootstrap_index(5)
    assert idx is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert len(idx) == 5, "ต้องได้ 5 ตำแหน่ง แต่ได้ %d" % len(idx)
    assert idx.min() >= 0 and idx.max() <= 4, "ตำแหน่งต้องอยู่ในช่วง 0 ถึง 4"
    assert np.issubdtype(np.asarray(idx).dtype, np.integer), "ต้องเป็นจำนวนเต็ม"


def test_bootstrap_index_has_repeats():
    np.random.seed(0)
    saw_repeat = False
    for _ in range(20):
        idx = bootstrap_index(10)
        if len(np.unique(idx)) < 10:
            saw_repeat = True
    assert saw_repeat, "สุ่มแบบคืนที่ต้องมีตัวซ้ำบ้าง (เผลอใช้ replace=False หรือเปล่า)"


def test_bootstrap_index_applies_to_both():
    np.random.seed(1)
    X = np.arange(12.).reshape(2, 6)
    Y = np.array([0., 1., 2., 3., 4., 5.])
    idx = bootstrap_index(6)
    assert X[:, idx].shape == (2, 6) and Y[idx].shape == (6,)
    assert np.allclose(X[0, idx], Y[idx]), \
        "ถ้าใช้ idx ชุดเดียวกัน ข้อมูลกับ label จะยังตรงคู่กัน"


grade("7.1 bootstrap_index", 4,
      test_bootstrap_index, test_bootstrap_index_has_repeats, test_bootstrap_index_applies_to_both)

[PASS] 7.1 bootstrap_index                4 / 4


### 7.2 `choose_sqrt` (4 คะแนน)

สุ่มเลือก $m = \lfloor\sqrt{p}\rfloor$ ตำแหน่งแบบไม่คืนที่

**ตัวอย่างที่คำนวณให้แล้ว**

```
p = 9   ->  m = floor(sqrt(9))  = floor(3.0)  = 3   ->  เช่น [7, 2, 5]
p = 4   ->  m = floor(sqrt(4))  = 2                 ->  เช่น [3, 0]
p = 2   ->  m = floor(sqrt(2))  = floor(1.414) = 1  ->  เช่น [1]
p = 16  ->  m = 4
```

**คำใบ้**
- `int(np.floor(np.sqrt(p)))` ได้ `m`
- `np.random.choice(p, m, replace=False)` ได้ตำแหน่ง

In [74]:
def choose_sqrt(p):
    '''
    สุ่มเลือก floor(sqrt(p)) ตำแหน่งจากช่วง 0 ถึง p-1 แบบไม่คืนที่
    ใส่เข้า : p เป็นจำนวนเต็ม
    ได้ออก  : idx ขนาด (m,) โดย m = floor(sqrt(p)) ไม่มีค่าซ้ำ
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    m = int(np.floor(np.sqrt(p)))
    idx = np.random.choice(p, m, replace=False)
    #########################################
    return idx

In [75]:
def test_choose_sqrt_size():
    np.random.seed(0)
    idx = choose_sqrt(9)
    assert idx is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert len(idx) == 3, "p=9 ต้องได้ 3 ตำแหน่ง แต่ได้ %d" % len(idx)
    assert len(choose_sqrt(4)) == 2, "p=4 ต้องได้ 2"
    assert len(choose_sqrt(2)) == 1, "p=2 ต้องได้ 1 เพราะ floor(1.414) = 1"
    assert len(choose_sqrt(16)) == 4


def test_choose_sqrt_no_repeat():
    np.random.seed(0)
    for _ in range(20):
        idx = choose_sqrt(16)
        assert len(np.unique(idx)) == len(idx), \
            "ห้ามซ้ำ ต้องใส่ replace=False"
        assert idx.min() >= 0 and idx.max() <= 15


def test_choose_sqrt_is_random():
    np.random.seed(0)
    seen = set()
    for _ in range(40):
        seen.add(tuple(sorted(choose_sqrt(9))))
    assert len(seen) > 3, "ต้องสุ่มจริง ไม่ใช่คืนชุดเดิมทุกครั้ง"


grade("7.2 choose_sqrt", 4,
      test_choose_sqrt_size, test_choose_sqrt_no_repeat, test_choose_sqrt_is_random)

[PASS] 7.2 choose_sqrt                    4 / 4


### 7.3 `flip_coin` (4 คะแนน)

โยนเหรียญที่ออกหัวด้วยความน่าจะเป็น `e`

**ตัวอย่างที่คำนวณให้แล้ว** `np.random.rand()` สุ่มจำนวนจริงจาก 0 ถึง 1 แบบสม่ำเสมอ

```
สุ่มได้ 0.0374  ->  0.0374 < 0.1  เป็นจริง   ->  True
สุ่มได้ 0.9507  ->  0.9507 < 0.1  เป็นเท็จ   ->  False
สุ่มได้ 0.0580  ->  True
```

เหตุผลว่าทำไมได้ความน่าจะเป็น `e` พอดี เพราะค่าที่สุ่มกระจายสม่ำเสมอบนช่วง $[0, 1)$
สัดส่วนของช่วงที่น้อยกว่า `e` ก็คือความยาว `e` เอง ถ้าโยน 1000 ครั้งด้วย `e = 0.1`
จะได้ `True` ประมาณ 100 ครั้ง

**สังเกตสองอย่าง** `np.random.rand()` ไม่ต้องใส่อาร์กิวเมนต์ จะได้ตัวเลขตัวเดียว
ไม่ใช่ array และใช้ `<` ไม่ใช่ `<=` เพื่อให้ `e = 0` แปลว่าไม่เคยจริงเลยจริง ๆ

**คำใบ้** บรรทัดเดียว `np.random.rand() < e`

In [76]:
def flip_coin(e):
    '''
    คืนค่า True ด้วยความน่าจะเป็น e
    ใส่เข้า : e เป็นจำนวนจริงระหว่าง 0 ถึง 1
    ได้ออก  : out เป็น True หรือ False
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้
    out = np.random.rand() < e
    #########################################
    return out

In [77]:
def test_flip_coin_rate():
    np.random.seed(0)
    n = 2000
    count = sum(flip_coin(0.1) for _ in range(n))
    assert count is not None
    assert 140 <= count <= 260, \
        "e=0.1 โยน 2000 ครั้ง ควรได้ True ประมาณ 200 ครั้ง แต่ได้ %d" % count

    np.random.seed(0)
    count = sum(flip_coin(0.5) for _ in range(n))
    assert 900 <= count <= 1100, "e=0.5 ควรได้ประมาณครึ่งหนึ่ง แต่ได้ %d" % count


def test_flip_coin_edges():
    np.random.seed(0)
    assert not any(flip_coin(0.0) for _ in range(500)), \
        "e=0 ต้องไม่เป็นจริงเลย ใช้ < ไม่ใช่ <= หรือเปล่า"
    assert all(flip_coin(1.0) for _ in range(500)), "e=1 ต้องเป็นจริงทุกครั้ง"


grade("7.3 flip_coin", 4, test_flip_coin_rate, test_flip_coin_edges)

[PASS] 7.3 flip_coin                      4 / 4


---
# 8. ประกอบร่าง (ไม่คิดคะแนน)

ข้อนี้ไม่มีคะแนน แต่เป็นข้อที่สำคัญที่สุดในแลป เพราะมันคือ **ข้อแรกของ Lab 1** จริง ๆ

คุณจะเห็นว่าสิ่งที่ดูยากในสัปดาห์หน้า แท้จริงคือสามบรรทัดที่คุณเพิ่งฝึกมาแล้วทั้งหมด

### โจทย์

หาค่า $\mathbf{w}$ ที่ทำให้ผลรวมกำลังสองของความคลาดเคลื่อนน้อยที่สุด คำตอบคือ normal equation

$$\mathbf{w} = (X^\top X)^{-1} X^\top \mathbf{y}$$

เขียนใหม่ในรูปที่เอาไปสั่งคอมพิวเตอร์ได้ แทนที่จะหา inverse ตรง ๆ เราแก้ระบบสมการแทน

$$(X^\top X)\,\mathbf{w} = X^\top \mathbf{y}$$

ซึ่งอยู่ในรูป $A\mathbf{z} = \mathbf{b}$ พอดี โดย $A = X^\top X$ และ $\mathbf{b} = X^\top \mathbf{y}$

### ตัวอย่างที่คำนวณให้แล้ว

$$X = \begin{bmatrix}1 & -1\\ 1 & 0\\ 1 & 1\end{bmatrix},
\qquad \mathbf{y} = \begin{bmatrix}1.5\\ 2.5\\ 3.5\end{bmatrix}$$

คอลัมน์แรกเป็น 1 ทั้งหมด คือคอลัมน์ bias จากข้อ 2.3 ส่วนคอลัมน์ที่สองคือ feature จริง

$$X^\top X = \begin{bmatrix}3 & 0\\ 0 & 2\end{bmatrix},
\qquad X^\top \mathbf{y} = \begin{bmatrix}1.5 + 2.5 + 3.5\\ -1.5 + 0 + 3.5\end{bmatrix}
= \begin{bmatrix}7.5\\ 2\end{bmatrix}$$

แก้สมการได้

$$3w_0 = 7.5 \;\Rightarrow\; w_0 = 2.5
\qquad
2w_1 = 2 \;\Rightarrow\; w_1 = 1$$

$$\mathbf{w} = \begin{bmatrix}2.5\\ 1\end{bmatrix}$$

แปลว่าเส้นที่พอดีที่สุดคือ $\hat{y} = 2.5 + 1 \cdot x$ ลองแทนค่าดู $x = -1$ ได้ 1.5 ถูกต้อง
$x = 0$ ได้ 2.5 ถูกต้อง $x = 1$ ได้ 3.5 ถูกต้อง ข้อมูลชุดนี้อยู่บนเส้นตรงพอดี

### งานของคุณ

เขียนสองบรรทัด โดยใช้สิ่งที่ทำมาแล้วในข้อ 6.2 และ 6.4

In [78]:
def least_square(X, y):
    '''
    หาค่า w ด้วยวิธีกำลังสองน้อยที่สุด
    ใส่เข้า : X ขนาด (n, p), y ขนาด (n, 1)
    ได้ออก  : w ขนาด (p, 1)

    คำใบ้  บรรทัดแรกสร้าง A กับ b  บรรทัดที่สองเรียก np.linalg.solve
    '''
    #########################################
    ## เขียนโค้ดของคุณตรงนี้ / INSERT YOUR CODE HERE
    A, b = X.T @ X, X.T @ y
    w = np.linalg.solve(A, b)
    #########################################
    return w

In [79]:
def test_least_square():
    X = np.array([[1., -1.],
                  [1., 0.],
                  [1., 1.]])
    y = np.array([[1.5], [2.5], [3.5]])
    w = least_square(X, y)
    assert w is not None, "ยังไม่ได้เขียนโค้ด ฟังก์ชันคืนค่า None"
    assert w.shape == (2, 1), "ต้องได้ (2, 1) แต่ได้ %s" % (w.shape,)
    assert np.allclose(w, [[2.5], [1.]], atol=1e-6)


def test_least_square_recovers_the_truth():
    '''สร้างข้อมูลจากเส้นที่เรารู้คำตอบอยู่แล้ว แล้วดูว่าหาคำตอบนั้นกลับมาได้ไหม'''
    rng = np.random.default_rng(0)
    n = 300
    x = rng.random((n, 1)) * 10
    X = np.hstack([np.ones((n, 1)), x])          # ข้อ 2.3
    w_true = np.array([[-2.], [0.7]])
    y = X @ w_true + rng.normal(0., 0.05, (n, 1))

    w = least_square(X, y)
    assert np.allclose(w, w_true, atol=0.05), \
        "ควรหาค่าจริง [-2, 0.7] กลับมาได้ แต่ได้ %s" % w.reshape(-1)
    print("         w จริง     ", w_true.reshape(-1))
    print("         w ที่หาได้  ", w.reshape(-1))


grade("8.1 least_square (ไม่คิดคะแนน)", 0,
      test_least_square, test_least_square_recovers_the_truth)

         w จริง      [-2.   0.7]
         w ที่หาได้   [-1.9969  0.6995]
[PASS] 8.1 least_square (ไม่คิดคะแนน)     0 / 0


## คะแนนของคุณ

In [80]:
report()

                                        score
----------------------------------------------------
1    สร้าง array และรูปร่าง               24 / 24  
2    elementwise และ broadcasting         28 / 28  
3    รวมค่าด้วย axis                      16 / 16  
4    boolean mask                         20 / 20  
5    unique, argmax, argsort              16 / 16  
6    พีชคณิตเชิงเส้น                      20 / 20  
7    การสุ่ม                              12 / 12  
8    ประกอบร่าง                            0 / 0   
----------------------------------------------------
TOTAL                                    136 / 136 


---
## สามปัญหาที่จะเจอซ้ำทั้งเทอม

เขียนไว้ตรงนี้เพื่อให้กลับมาอ่านตอนติด

**หนึ่ง รูปร่างผิดแต่ไม่ error** `(n,)` กับ `(n, 1)` ต่างกัน และเวลาเอามาคูณกันมันไม่ฟ้อง
แต่ได้ตารางขนาด `(n, n)` ออกมาแทน ถ้าเห็นตัวเลขเยอะผิดปกติ หรือโปรแกรมช้าผิดปกติ
ให้พิมพ์ `.shape` ของทุกตัวแปรออกมาดูก่อนเสมอ

**สอง `axis` คือแกนที่ยุบหายไป** ไม่ใช่แกนที่เก็บไว้ ถ้าไม่แน่ใจ ให้ลองกับ array เล็ก ๆ
ที่คำนวณด้วยหัวได้ แล้วดูว่าได้ตรงกับที่คิดไหม อย่าเดา

**สาม `*` ไม่ใช่การคูณเมทริกซ์** ใช้ `@` สำหรับคูณเมทริกซ์เสมอ ถ้าเขียน `A * B`
แล้วรูปร่างบังเอิญเข้ากันได้ มันจะคูณทีละตัวให้โดยไม่บอกว่าคุณตั้งใจอีกอย่าง

สามข้อนี้คือสาเหตุของบั๊กเกือบทั้งหมดที่จะเจอ ไม่ใช่สูตรผิด